# Lund distribution closure — production test v0

[`lund_distribution_closure_v2.ipynb`](lund_distribution_closure_v2.ipynb), run against the **held-out** file of
[`docs/PLAN_prod_test_v0.md`](../docs/PLAN_prod_test_v0.md) with every setting already applied.

**Generated** by [`scripts/make_prod_closure_nb.py`](../scripts/make_prod_closure_nb.py); every cell below
section 0 is byte-identical to v2. Edit v2 (or the generator) and regenerate — a hand-edited
copy would be a second definition of the same headline ratios, which is exactly how two
closure populations drifted apart before.

Five settings differ from v2's defaults, and they are read from `prod_test_v0_metrics.json`
rather than pasted in, so they cannot disagree with the fit that produced them:

| constant | why it must change |
|---|---|
| `CKPT_PATH` | the production-test checkpoint |
| `ROOT_PATH` | the independent test file — v2 defaults to `cpp/test_data/jets.root` |
| `EMPTY_THRESHOLD` | the **frozen** tau from the training val split; v2's `None` rate-matches on the sample it reports on, which is circular |
| `LENGTH_TEMPERATURE` | fitted on the training val split; reaches `length_pmf` **and** `sample` |
| `LENGTH_TILT` | the same fit. A scalar temperature is symmetric about the mode, so it cannot move `q(0\|x)` the way a monotone ramp in `n` requires — whenever the head needs correcting at all |

Run [`notebooks/prod_test_v0.ipynb`](prod_test_v0.ipynb) first; this reads its artifact.


## 0. Parameters

In [ ]:
# ===========================================================================
# GENERATED — do not hand-edit. Regenerate with:
#     python scripts/make_prod_closure_nb.py
# Every cell below section 0 is byte-identical to
# notebooks/lund_distribution_closure_v2.ipynb; only this cell differs, so the two
# notebooks cannot drift into two definitions of the same headline number.
# ===========================================================================
#
# The five production-test settings are READ FROM THE RUN'S OWN ARTIFACT rather than
# pasted in, so the frozen `tau` and the fitted `(temperature, tilt)` cannot disagree
# with the section-6 fit that produced them. Run notebooks/prod_test_v0.ipynb first.
import json as _json
from pathlib import Path as _Path

# None -> newest runs/prod_test_v0/*/prod_test_v0/prod_test_v0_metrics.json
PROD_METRICS_PATH = None

_REPO = _Path.cwd().parent if _Path.cwd().name == "notebooks" else _Path.cwd()
if PROD_METRICS_PATH:
    _mp = _Path(PROD_METRICS_PATH)
    _mp = _mp if _mp.is_absolute() else _REPO / _mp
else:
    _found = sorted(_REPO.glob("runs/prod_test_v0/*/prod_test_v0/prod_test_v0_metrics.json"),
                    key=lambda q: q.stat().st_mtime)
    if not _found:
        raise FileNotFoundError(
            "no prod_test_v0_metrics.json under runs/prod_test_v0/. This notebook takes "
            "its checkpoint, its test file, the frozen empty-tree tau and the fitted "
            "length recalibration from that artifact — run notebooks/prod_test_v0.ipynb "
            "first, or set PROD_METRICS_PATH."
        )
    _mp = _found[-1]

_M = _json.loads(_mp.read_text())
# Read the PRIMARY record of each value, not a summary block duplicating it: the tau and
# the (T, tilt) live where section 6 fitted them, and a second copy is a second thing that
# can be stale. This also means any prod_test_v0 artifact works, including ones written
# before this notebook existed.
try:
    _IN = {
        "CKPT_PATH": _M["run"]["checkpoint"],
        "ROOT_PATH": _M["run"]["test_path"],
        "EMPTY_THRESHOLD": _M["empty_tree"]["tau"]["value"],
        "LENGTH_TEMPERATURE": _M["empty_tree"]["recalibration"]["T"],
        "LENGTH_TILT": _M["empty_tree"]["recalibration"]["tilt"],
    }
except KeyError as _e:
    raise KeyError(
        f"{_mp} has no {_e} — it is not a prod_test_v0 metrics file, or predates the "
        f"section-6 recalibration. Re-run notebooks/prod_test_v0.ipynb."
    ) from None
print(f"[prod_test_v0] settings from {_mp.relative_to(_REPO)}")
for _k, _v in _IN.items():
    print(f"    {_k:<20} = {_v!r}")

# --- inputs -----------------------------------------------------------------
CKPT_PATH   = _IN["CKPT_PATH"]
#             repo-relative; None -> newest runs/**/best.ckpt
ROOT_PATH   = _IN["ROOT_PATH"]            # the INDEPENDENT file, not the trained-on one
NTUPLE_NAME = "Jets"

# --- sample selection -------------------------------------------------------
PT_VAR  = "jet_pt"    # "jet_pt" (ungroomed) | "x_ptg" (groomed)
PT_MIN  = None        # half-open [PT_MIN, PT_MAX) GeV; both None -> keep every jet
PT_MAX  = None
# THE v2 CHANGE. False -> select len(x)>0 only, the deployable population: every jet
# you could pick out on data, including the ~17% whose parton truth is the empty tree.
# True -> also require len(y)>0, reproducing v1's population exactly. That condition
# reads the answer, so it is not a selection any analysis can make.
REQUIRE_TRUTH_SPLITTING = False
N_JETS  = 2000        # jets in the evaluation pass  (run the cost probe first)
SEED    = 1234
DEVICE  = "cpu"       # keep "cpu": this model is tiny (~120k params) and decoded one
#                       jet at a time, so a GPU never amortises its dispatch overhead.
#                       Measured on M-series MPS: every torch stage 10-15x SLOWER than
#                       CPU, ~1.5x slower overall. "auto"/"mps"/"cuda" work, but MPS
#                       cannot help here and is not worth the nondeterminism.

# --- inference knobs --------------------------------------------------------
K_DRAWS               = 120   # posterior draws per jet, shared by floor / MBR / posterior
LENGTH_FLOOR_QUANTILE = 0.15  # per-jet MAP floor at this quantile of P(n|x); 0.0 -> off
# MBR is ~96% of the runtime, so this is THE speed knob. Measured per jet at
# K_DRAWS=120, MBR_N_CANDIDATES=16 (total pass incl. sampling and decode):
#   "pot"        113 ms  -- exact OT, no extra dependency, the DecodeConfig default
#   "energyflow"  19 ms  -- the SAME perturbative-Lund EMD, 6x faster overall;
#                           picks an identical MBR tree on 99.3% of jets (100% on
#                           multiplicity), the rest being solver tie-breaks. Switch
#                           to this if energyflow is installed -- it is the same
#                           number, not an approximation.
#   "surrogate"   10 ms  -- a binned Lund-image chi2: a DIFFERENT risk function, so
#                           it picks a different tree on ~87% of jets. Fine for
#                           iterating on the figures, never for reported numbers.
MBR_BACKEND           = "pot"
MBR_N_CANDIDATES      = 16     # MBR candidate cap per jet (0 = all K draws)
# Re-decode the MAP a second time with the length floor lifted (min_emissions=0), so
# section 6 can price what the floor costs on jets whose truth IS the empty tree. The
# floored MAP stays the headline estimator; this is the control, not a replacement.
MAP_ALLOW_EMPTY       = True
# decode.empty_threshold: decide the EMPTY tree when q(N=0|x) >= tau, BEFORE any shape
# decode (docs/PLAN_empty_parton_tree.md). None -> rate-matched on THIS sample, which is
# what makes the row comparable to truth here; 0.0 -> off. In production fit it on
# held-out jets with `empty_threshold_for_rate` and FREEZE it -- it is a quantile, so it
# is sample-dependent and must be re-fitted per pT window. Rate-matching on the SAME jets
# you report on reproduces the fitted rate by construction, so it measures the quantile
# function rather than the model: for a real held-out assessment, pass the frozen tau.
EMPTY_THRESHOLD       = float(_IN["EMPTY_THRESHOLD"])   # FROZEN, not rate-matched here
# Post-hoc recalibration of the length head, `softmax(log q(n|x)/T + tilt*n)`. These reach
# `length_pmf` AND `sample`, so they move the posterior series, the empty rate and TAU
# above -- which is exactly why they are constants here and are recorded in the artifact:
# an empty rate whose (T, tilt) is unknown is unattributable. Fit them on HELD-OUT jets
# with `inference.length.fit_length_recalibration` and freeze, never on this sample.
# (1.0, 0.0) is the identity and reproduces the checkpoint's own decode exactly.
# None -> take whatever the checkpoint snapshot carries.
LENGTH_TEMPERATURE    = float(_IN["LENGTH_TEMPERATURE"])
LENGTH_TILT           = float(_IN["LENGTH_TILT"])

# --- binning / figures ------------------------------------------------------
# Coarsening factor per observable, applied to the C++ app's edges. Every value
# must divide the app's bin count, so the edges stay a strict SUBSET of
# hist_lund_rntuple.cpp's -- any panel here still overlays lund_rntuple_histograms.ipynb
# bin for bin.
REBIN    = {"lnInvDelta": 2, "lnkt": 2, "lnz": 1, "psi": 4, "mult": 1}
T_SLICES = (0, 1, 2, 3)   # per-splitting-index panels; a "4+" pool is appended
T_LADDER = 10             # ladder profiles run t = 0 .. T_LADDER-1
PLANE_NB = 30             # Lund-plane bins per axis (multiple of geometry.n_bins)

# --- scoreability gate ------------------------------------------------------
# An improvement ratio only means something when the baseline it divides by is
# bigger than the noise. These control the jet-level bootstrap that measures that
# noise floor; rows whose plain-RSD distance falls inside it are reported but not
# scored. See section 12.
N_BOOT    = 24            # bootstrap resamples per observable
FLOOR_PCT = 95            # percentile of the null distances taken as the floor

WRITE_ARTIFACTS = True    # dist_closure_{metrics.json,table.md} beside the checkpoint

# --- what this variant relies on v2 already getting right --------------------
# Asserted, not assumed: these are v2 defaults today, and a future change to them would
# otherwise silently alter the production-test numbers instead of failing here.
assert MBR_BACKEND != "surrogate", (
    "the surrogate is a different risk function and, before the n_bins fix, a coarser "
    "one — no reported number may use it"
)
assert REQUIRE_TRUTH_SPLITTING is False, (
    "the production test reports on the DEPLOYABLE population (len(x) > 0); requiring a "
    "truth splitting selects on the answer"
)
assert PLANE_NB % 30 == 0, (
    f"PLANE_NB={PLANE_NB} must stay a multiple of geometry.n_bins (30) so the Lund plane "
    f"edges remain a strict subset of the model's own cells"
)
assert str(_IN["ROOT_PATH"]) != str(_M["run"]["train_path"]), (
    "the eval file is the file this checkpoint TRAINED on — that is not a closure test"
)

# THE scale check. `EMPTY_THRESHOLD` is a QUANTILE of q(0|x), so it only means anything
# on the distribution it was fitted to — and (T, tilt) move that distribution's mean by
# ~3x. Fitting on the raw head and applying here, where the head is recalibrated, leaves
# the RANKING untouched and the CUT in the wrong place: the rate goes to ~3x truth and
# precision collapses, with nothing in either notebook to say why.
_under = _M["empty_tree"]["tau"].get("fitted_under")
assert _under is not None, (
    "this artifact records no scale for its tau (prod_test_v0 predating the fix). "
    "Re-run notebooks/prod_test_v0.ipynb: a tau without its scale cannot be applied."
)
assert (abs(float(_under["length_temperature"]) - LENGTH_TEMPERATURE) < 1e-9
        and abs(float(_under["length_tilt"]) - LENGTH_TILT) < 1e-9), (
    f"EMPTY_THRESHOLD was fitted at (T, tilt) = "
    f"({_under['length_temperature']}, {_under['length_tilt']}) but this notebook applies "
    f"({LENGTH_TEMPERATURE}, {LENGTH_TILT}). A tau is a quantile of q(0|x); on a "
    f"different scale it cuts in the wrong place."
)


## 1. Imports, house style, binning, helpers

The style block, the `BINS` dict and the `edges`/`step`/`zoom`/`finish` helpers are
inherited from [`lund_rntuple_histograms.ipynb`](lund_rntuple_histograms.ipynb) so the two
notebooks are directly comparable — same edges, same neutrals, same sequential ramp.

**Series colours are not inherited from `inference_demo.ipynb`.** Its five-way
green/grey/red/purple/blue set fails colour-vision separation as a set (blue vs purple sits
at $\Delta E \approx 1.7$ under protanopia — indistinguishable). Here the two *data* series
take non-hue roles — truth is ink, plain RSD is a grey fill — which frees the three *model*
series to take the first three slots of the validated categorical palette, a set that
clears every all-pairs gate. The posterior series is additionally dashed, since it is a
reference rather than a contender.

In [ ]:
import math
import time
import warnings
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import torch
from omegaconf import OmegaConf
from scipy.stats import wasserstein_distance

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
warnings.filterwarnings("ignore", category=UserWarning)

from h2p_rsd_junipr.config import decode_params
from h2p_rsd_junipr.data.datamodule import select_pt_range
from h2p_rsd_junipr.data.dataset import MatchedLundDataset
from h2p_rsd_junipr.data.rntuple import load_rntuple
from h2p_rsd_junipr.data.synthetic import synthetic_matched_dataset
from h2p_rsd_junipr.eval.calibration import REGION_LABELS, cell_region
from h2p_rsd_junipr.eval.report import save_metrics
from h2p_rsd_junipr.features import node_raw
from h2p_rsd_junipr.geometry import Geometry
from h2p_rsd_junipr.inference.length import (
    empty_threshold_for_rate,
    learned_min_emissions,
)
from h2p_rsd_junipr.models.ar_junipr import ARJunipr
from h2p_rsd_junipr.models.base import build_model
from h2p_rsd_junipr.train.checkpoint import load_for_inference
from h2p_rsd_junipr.train.trainer import seed_everything, select_device

# --- style -------------------------------------------------------------------
# Roles, not a flat palette. truth = ink (the reference every distance is measured
# against), plain RSD = grey fill (the do-nothing backdrop). Those two carry no hue,
# so the three MODEL series get the first three slots of the validated categorical
# palette -- the only three that clear the all-pairs colour-vision gates, which is
# what overlaid step histograms need since every pair is visually adjacent.
SURFACE, INK, INK_2, MUTED, GRID, AXIS = (
    "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7",
)
C_TRUTH = INK          # truth y            -- ink, thick solid
C_RSD_F, C_RSD_E = "#e1e0d9", "#898781"   # plain-RSD x -- grey fill + edge
C_MAP   = "#2a78d6"    # MAP point estimate -- blue   (slot 1)
C_MBR   = "#eb6834"    # MBR point estimate -- orange (slot 2)
C_POST  = "#199e70"    # posterior draw     -- aqua   (slot 3), dashed

SEQ_BLUE = [
    "#cde2fb", "#b7d3f6", "#9ec5f4", "#86b6ef", "#6da7ec", "#5598e7", "#3987e5",
    "#2a78d6", "#256abf", "#1c5cab", "#184f95", "#104281", "#0d366b",
]
CMAP = mpl.colors.LinearSegmentedColormap.from_list("h2p_blue", SEQ_BLUE)
CMAP.set_bad(SURFACE)   # empty bins recede to the surface instead of reading as data
# Ratio maps are polarity, not magnitude: two poles + a NEUTRAL grey midpoint, so
# "agrees with truth" reads as nothing at all.
DIV = mpl.colors.LinearSegmentedColormap.from_list(
    "h2p_div", ["#0d366b", "#2a78d6", "#9ec5f4", "#f0efec", "#f0a3a3", "#d03b3b", "#7a1f1f"]
)
DIV.set_bad(SURFACE)

mpl.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 120,
    "figure.facecolor": SURFACE, "savefig.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "axes.edgecolor": AXIS, "axes.linewidth": 0.8,
    "axes.labelcolor": INK_2, "axes.titlecolor": INK,
    "axes.titlesize": 9, "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True,
    "text.color": INK, "xtick.color": MUTED, "ytick.color": MUTED,
    "xtick.labelcolor": INK_2, "ytick.labelcolor": INK_2,
    "grid.color": GRID, "grid.linestyle": "-", "grid.linewidth": 0.6,
    "font.size": 9, "legend.frameon": False, "lines.linewidth": 1.6,
})

# --- binning: hist_lund_rntuple.cpp, coarsened by REBIN ----------------------
BINS = {
    "lnInvDelta": (100, 0.0, 10.0),
    "lnkt": (120, -4.0, 8.0),
    "lnz": (100, -10.0, 0.0),
    "psi": (100, -np.pi, np.pi),
    "mult": (51, -0.5, 50.5),
}
LABEL = {
    "lnInvDelta": r"$\ln(1/\Delta R)$",
    "lnkt": r"$\ln(k_t/\mathrm{GeV})$",
    "lnz": r"$\ln z$",
    "psi": r"$\psi$",
    "mult": "primary splittings / jet",
}
COL = {"lnInvDelta": 0, "lnkt": 1, "lnz": 2, "psi": 3}   # node_raw column order


def edges(key):
    """The app's edges coarsened by REBIN[key] -- a strict subset of them."""
    n, lo, hi = BINS[key]
    r = int(REBIN.get(key, 1))
    if n % r:
        raise ValueError(f"REBIN[{key}]={r} does not divide the app's {n} bins")
    return np.linspace(lo, hi, n // r + 1)


def h1(values, weights, e):
    return np.histogram(values, bins=e, weights=weights)[0]


def h1_sumw2(values, weights, e, w2=None):
    """Weighted counts and their Sumw2 errors -- ROOT's TH1::Sumw2 convention.

    `w2` overrides the sum-of-squares term. It exists for the bootstrap in section
    12, where an entry standing in for `c` independent copies contributes `c*w**2`,
    not `(c*w)**2` -- the default is right whenever one row means one entry.
    """
    c = np.histogram(values, bins=e, weights=weights)[0]
    sq = np.asarray(weights, float) ** 2 if w2 is None else w2
    s2 = np.histogram(values, bins=e, weights=sq)[0]
    return c, np.sqrt(s2)


def density(counts, err, e):
    """Unit-area normalisation with the error propagated through it.

    The overall scale is treated as fixed (the usual shape-comparison convention);
    the residual correlation it induces between bins is second order and is not
    modelled here -- which is one reason chi2/ndf is reported next to W1 and KS
    rather than on its own.
    """
    w = np.diff(e)
    tot = float((counts * w).sum())
    if tot <= 0:
        return np.zeros_like(counts, dtype=float), np.zeros_like(counts, dtype=float)
    return counts / tot, err / tot


def step(ax, y, e, color, label=None, lw=1.6, ls="-", z=3):
    ax.stairs(y, e, color=color, linewidth=lw, linestyle=ls, label=label, zorder=z)


def fill(ax, y, e, face, edge, label=None, z=1):
    ax.stairs(y, e, color=edge, fill=True, facecolor=face, linewidth=1.0,
              label=label, zorder=z)


def zoom(ax, ys, e, pad=0.03):
    """Keep the app's binning, view only the populated range (a view limit only)."""
    tot = np.sum(np.atleast_2d(np.asarray(ys, dtype=float)), axis=0)
    hit = np.flatnonzero(tot > 0)
    if hit.size == 0:
        return
    lo, hi = e[hit[0]], e[hit[-1] + 1]
    m = pad * (hi - lo)
    ax.set_xlim(lo - m, hi + m)


def finish(ax, xlabel="", title="", ylabel="", logy=False, legend=False):
    if logy:
        ax.set_yscale("log")
    if xlabel:
        ax.set_xlabel(xlabel)
    if ylabel:
        ax.set_ylabel(ylabel)
    if title:
        ax.set_title(title)
    if legend:
        ax.legend(fontsize=7.5, loc="best")


def ratio_axes(n_cols, n_rows=1, w=4.1, h=3.4):
    """A grid of (main, ratio-to-truth) panel pairs -- the standard HEP layout.

    Nested gridspecs, because the two gaps are not the same gap: a main panel and
    its ratio strip must sit flush (they share an x axis and read as one figure),
    while consecutive ROWS need room for the next title. One uniform hspace cannot
    do both, and stacking rows with it collides titles into the strip above.
    """
    fig = plt.figure(figsize=(w * n_cols, h * n_rows))
    outer = fig.add_gridspec(n_rows, n_cols, hspace=0.46, wspace=0.28)
    pairs = []
    for r in range(n_rows):
        row = []
        for c in range(n_cols):
            inner = outer[r, c].subgridspec(2, 1, height_ratios=[3, 1], hspace=0.07)
            a = fig.add_subplot(inner[0])
            b = fig.add_subplot(inner[1], sharex=a)
            a.tick_params(labelbottom=False)
            b.axhline(1.0, color=MUTED, lw=0.8, zorder=1)
            b.set_ylim(0.0, 2.0)
            b.set_ylabel("/ truth", fontsize=7.5)
            row.append((a, b))
        pairs.append(row)
    return fig, pairs


def ratio_of(num, den):
    """Bin-wise ratio, masked where truth has no support (0/0 is not 1)."""
    out = np.full_like(np.asarray(num, dtype=float), np.nan)
    ok = np.asarray(den, dtype=float) > 0
    out[ok] = np.asarray(num, dtype=float)[ok] / np.asarray(den, dtype=float)[ok]
    return out


def fig_legend(fig, ax, title, ncols=5, top=0.80):
    """One legend per figure, in reserved space -- never on top of the data.

    With five overlaid series there is no reliable free corner inside an axes, so
    `loc="best"` collides sooner or later; reserving a strip is deterministic.
    """
    h, lab = ax.get_legend_handles_labels()
    fig.subplots_adjust(top=top)
    fig.suptitle(title, x=0.006, y=1.005, ha="left")
    fig.legend(h, lab, loc="upper left", bbox_to_anchor=(0.006, 0.955),
               ncols=ncols, fontsize=8, frameon=False)


print("style, binning and helpers ready")

## 2. Load the trained model

Everything structural comes from the checkpoint's own config snapshot — geometry, encoder,
model family, and the aux conditioning columns the encoder was *built* for. That last one
matters: `inference_demo.ipynb` builds its dataset with no aux columns, which silently
mis-conditions an aux checkpoint. Here the dataset is built from `model.aux_feature_names`.

In [ ]:
def find_latest_checkpoint(runs_dir: Path):
    """Newest **/best.ckpt (falls back to last.ckpt). rglob, so nested run dirs
    like runs/calibration_v2_walkthrough/ar_junipr_v3/ are found too."""
    cks = sorted(runs_dir.rglob("best.ckpt"), key=lambda q: q.stat().st_mtime)
    if not cks:
        cks = sorted(runs_dir.rglob("last.ckpt"), key=lambda q: q.stat().st_mtime)
    return cks[-1] if cks else None


seed_everything(SEED)
device = select_device() if DEVICE == "auto" else torch.device(DEVICE)

CKPT = (REPO / CKPT_PATH) if CKPT_PATH else find_latest_checkpoint(REPO / "runs")
if CKPT is None:
    raise FileNotFoundError(f"no checkpoint under {REPO / 'runs'} -- train one first")

info = load_for_inference(CKPT)
cfg = OmegaConf.create(info["config"])
geom = Geometry.from_config(cfg.geometry)
model = build_model(cfg, geom).to(device)
model.load_state_dict(info["model_state"])
model.eval()

DECODE = decode_params(cfg)
# Length recalibration: the notebook constants win over the snapshot when set, mirroring
# how `cli.py::cmd_eval` re-applies a lifted `decode.length_temperature`. Setting the
# model attributes is what makes them reach `length_pmf` and `sample`; writing them back
# into DECODE is what makes them appear in `dist_closure_metrics.json`.
if LENGTH_TEMPERATURE is not None:
    DECODE["length_temperature"] = float(LENGTH_TEMPERATURE)
if LENGTH_TILT is not None:
    DECODE["length_tilt"] = float(LENGTH_TILT)
model.length_temperature = float(DECODE["length_temperature"])
model.length_tilt = float(DECODE["length_tilt"])
if (model.length_temperature, model.length_tilt) != (1.0, 0.0):
    print(f"[decode] length head recalibrated: T = {model.length_temperature:.4f}, "
          f"tilt = {model.length_tilt:+.4f} -- the posterior series, the empty rate and "
          f"TAU below all move with these.")
BEAM = {k: DECODE[k] for k in ARJunipr._BEAM_KEYS if k in DECODE}
AUX = tuple(model.aux_feature_names)
# Does this family have a continuous coordinate density? This is the contract flag --
# every family that can draw ln z / psi sets it, and only ar_junipr_v1 does not. When
# it is False the nodes carry ln_z = 0, psi = 0 PLACEHOLDERS (z = 1: the softer prong
# taking the whole jet, which is not a physical configuration at all), so those panels
# must not plot them as predictions.
CONT = bool(getattr(model, "has_continuous_coords", False))

try:   # a checkpoint outside the repo is legitimate; just print it whole
    _ck = CKPT.resolve().relative_to(REPO.resolve())
except ValueError:
    _ck = CKPT
print(f"checkpoint : {_ck}")
print(f"model      : {info['model_name']}  +  {cfg.encoder.name}   "
      f"({sum(p.numel() for p in model.parameters()):,} params)")
print(f"epoch      : {info['epoch']}   best val NLL {info['best_val_nll']:.4f}")
print(f"geometry   : n_bins={geom.n_bins} ({geom.n_cells} cells)  "
      f"ln(1/dR) in {geom.ln_invdelta_range}  ln kt in {geom.ln_kt_range}")
print(f"device     : {device}")
print(f"aux inputs : {AUX if AUX else '(none)'}")
print(f"exact NLL  : {model.exact_likelihood}")
if not CONT:
    print("NOTE: this family has no continuous coordinate head -- point estimates sit at\n"
          "      cell centres with ln z = 0, psi = 0 PLACEHOLDERS. The ln z and psi panels\n"
          "      below drop the model series accordingly.")

## 3. Load the test sample

Real ROOT data is the point here — `plain RSD` *is* the file's `x_*` branches, and the
per-jet `weight` only exists there. The synthetic fallback keeps the notebook runnable, but
its jets are not groomed, so the soft-drop validity checks in §5 turn off.

In [ ]:
jets = load_rntuple(str(REPO / ROOT_PATH), NTUPLE_NAME) if ROOT_PATH else None
SOURCE = "rntuple"
if jets is None:
    SOURCE = "synthetic"
    print("no ROOT file -> synthetic fallback (no grooming provenance, all weights 1.0)")
    jets = synthetic_matched_dataset(max(4 * N_JETS, 2000), seed=SEED)

jets = select_pt_range(jets, var=PT_VAR, lo=PT_MIN, hi=PT_MAX)

# The v2 selection. len(x)>0 drops jets with no conditioning information at all, and is
# a cut any analysis can make on data. len(y)>0 reads the parton truth, so it is NOT --
# it is opt-in here only to reproduce v1.
_n_in = len(jets)
_x0 = sum(1 for j in jets if not len(j["x"][0]))
_y0_of_x = sum(1 for j in jets if len(j["x"][0]) and not len(j["y"][0]))
jets = [j for j in jets
        if len(j["x"][0]) and (len(j["y"][0]) or not REQUIRE_TRUTH_SPLITTING)]
if not jets:
    raise RuntimeError("no jets survived the selection")

try:
    ds = MatchedLundDataset(jets, geom, aux_features=AUX)
except Exception as exc:   # aux_vector rejects the absent-column sentinels
    raise RuntimeError(
        f"the checkpoint was trained with aux inputs {AUX} but {ROOT_PATH} cannot supply "
        f"them ({exc}). Point ROOT_PATH at a file written with the aux columns, e.g. "
        f"cpp/test_data/jets_aux.root."
    ) from exc

W_ALL = np.array([float(j.get("weight", 1.0)) for j in jets], dtype=float)
# Grooming provenance -- the soft-drop boundary in section 5 is read off these.
SD_KNOWN = "z_cut" in jets[0] and jets[0]["z_cut"] is not None
Z_CUT = float(jets[0]["z_cut"]) if SD_KNOWN else float("nan")
BETA = float(jets[0].get("beta", 0.0) or 0.0) if SD_KNOWN else float("nan")
# Global ln z floor implied by soft drop over the whole angular window:
#   z > z_cut (dR/R0)^beta   <=>   ln z > ln z_cut - beta * ln(1/dR)
LNZ_FLOOR = (math.log(Z_CUT) - BETA * geom.ln_invdelta_range[1]) if SD_KNOWN else -10.0

n_eff = float(W_ALL.sum() ** 2 / (W_ALL ** 2).sum())
print(f"source     : {SOURCE}  {ROOT_PATH if ROOT_PATH else ''}")
print(f"generator  : {jets[0].get('generator', 'n/a')}")
if SD_KNOWN:
    print(f"grooming   : z_cut={Z_CUT:.3f}  beta={BETA:.3f}  "
          f"kt_floor={jets[0].get('kt_floor', float('nan')):.3f} GeV   "
          f"=> ln z > {LNZ_FLOOR:.3f}")
else:
    print("grooming   : n/a (no provenance in this sample -- soft-drop checks disabled)")
if PT_MIN is not None or PT_MAX is not None:
    print(f"pT window  : {PT_VAR} in [{PT_MIN}, {PT_MAX})")
_mx = np.mean([len(j["x"][0]) for j in jets])
_my = np.mean([len(j["y"][0]) for j in jets])
_pz = np.mean([len(j["y"][0]) == 0 for j in jets])
print("selection  : len(x)>0"
      + ("  AND len(y)>0   [v1 population -- uses the truth]"
         if REQUIRE_TRUTH_SPLITTING else "   [deployable: no truth in the selection]"))
print(f"jets       : {len(jets):,} of {_n_in:,}   "
      f"(dropped {_n_in - len(jets):,}: {_x0:,} with no hadron splitting"
      + (f", {_y0_of_x:,} with no parton splitting)" if REQUIRE_TRUTH_SPLITTING
         else f"; {_y0_of_x:,} jets with no PARTON splitting are KEPT)"))
print(f"             sum(w)={W_ALL.sum():.6g}   effective N={n_eff:,.0f}")
print(f"mean mult  : hadron x {_mx:.3f}   parton y {_my:.3f}   x/y {_mx / _my:.3f}")
print(f"             P(n_y = 0) = {100 * _pz:.1f}%  <- the empty-tree rate section 6 scores")
if REQUIRE_TRUTH_SPLITTING:
    print("             WARNING: len(y)>0 is a truth-level cut with no data analogue. It")
    print("             removes the jets where hadronisation created every splitting, so")
    print("             the x/y excess above understates the real one (1.14 vs 1.31 on")
    print("             cpp/test_data/jets.root). Set REQUIRE_TRUTH_SPLITTING=False.")

## 4. The evaluation pass

One set of `K_DRAWS` posterior draws per jet feeds *three* consumers — the learned length
floor, the MBR risk minimisation, and the posterior-predictive series — so nothing is
sampled twice.

### Sampling the coordinate heads

`model.sample()` returns **cell chains only**, and a point estimate attaches head *modes*.
Neither is a genuine posterior draw in the continuous coordinates, and for $\ln z$ and
$\psi$ the mode is the *whole* story — so a mode-based "posterior" curve would carry
exactly the shrinkage it exists to expose.

`model.sample_coordinates(xf, nx, cells)` is the contract hook that draws them:
truncated normals for the within-cell offsets, a normal for $\ln z$, a von Mises for
$\psi$ in the AR case, and each other family's own sampler elsewhere. It returns `None`
only for a family with no coordinate density at all (`ar_junipr_v1`), and the cell-centre
fallback is then the honest answer rather than a filler constant plotted as a prediction —
`CONT` is what keeps those panels from lying.

In [ ]:
def pe_coords(pe):
    """LundPointEstimate -> (n, 4) in node_raw column order."""
    if not pe.nodes:
        return np.zeros((0, 4))
    return np.array([[n.ln_invDelta, n.ln_kt, n.ln_z, n.psi] for n in pe.nodes], dtype=float)


@torch.inference_mode()
def draw_coords(model, xf, nx, cells):
    """One posterior draw's CONTINUOUS coordinates, sampled rather than moded.

    `model.sample_coordinates` is the contract hook: `sample` returns cell chains only,
    and placing those at cell centres would leave ln z and psi holding a filler
    constant. Families without a coordinate density return None, and the cell-centre
    fallback is then the honest answer -- flagged by CONT, not silently plotted.

    Reproducibility rides on the global torch seed set by `seed_everything(SEED)`,
    since the contract signature takes no generator.
    """
    if not len(cells):
        return np.zeros((0, 4))
    c = model.sample_coordinates(xf, nx, list(cells))
    if c is None:
        return pe_coords(model.describe_cells(xf, nx, cells))
    return np.asarray(c.cpu().double().numpy(), dtype=float).reshape(-1, 4)


@torch.inference_mode()
def ar_kappa(model, xf, nx, cells):
    """Per-node von Mises concentration -- the psi caveat panel only, AR-only.

    `sample_coordinates` DRAWS coordinates and does not expose the head parameters,
    which is right: widening the contract for one diagnostic would be the wrong trade.
    So kappa alone is read off the AR heads, guarded, and the panel simply does not
    appear for other families.
    """
    L = len(cells)
    if L == 0 or not isinstance(model, ARJunipr) or not model.continuous_coords:
        return np.zeros(0)
    dev = xf.device
    e = model.encode(xf, nx)
    yc = torch.tensor([[int(c) for c in cells]], dtype=torch.long, device=dev)
    out = model._decode_states(yc, e, model.xattn_kv(xf, nx))
    eh = torch.cat([out, e.unsqueeze(1).expand(-1, L + 1, -1)], dim=-1)[:, :L, :]
    *_, kappa = model._coord_params(torch.cat([eh, model.y_embed(yc)], dim=-1))
    # .cpu() BEFORE .double(): MPS has no float64, so casting on-device raises.
    return kappa.squeeze(0).cpu().double().numpy()



SERIES = ("truth", "rsd", "map", "mbr", "post")
STYLE = {   # colour, line style, legend label
    "truth": (C_TRUTH, "-", r"truth $y$ (parton)"),
    "rsd":   (C_RSD_E, "-", r"plain RSD $x$ (hadron)"),
    "map":   (C_MAP,   "-", r"MAP $\hat y$"),
    "mbr":   (C_MBR,   "-", r"MBR $\hat y$"),
    "post":  (C_POST, "--", "posterior (1 draw/jet)" if CONT
                            else "posterior (cells sampled; no coordinate density)"),
}
MODELS = ("map", "mbr", "post")   # the series that get scored against plain RSD


def eval_jets(index):
    """The single pass. Returns per-jet (n,4) arrays for every series, + kappas.

    v2 also records `n_map0`, the MAP re-decoded with the length floor lifted, so
    section 6 can price what the floor costs on jets whose truth is the empty tree.
    """
    rng = np.random.default_rng(SEED)
    raw = {s: [] for s in SERIES}
    wjet, kappas, risks, n_map0 = [], [], [], []
    for i in index:
        item = ds[i]
        xf = item["xf"].unsqueeze(0).to(device)
        nx = torch.tensor([item["nx"]], device=device)

        draws = model.sample(xf, nx, n=K_DRAWS)
        mults = np.array([len(d) for d in draws], dtype=int)

        # learned per-jet MAP floor, reusing the draws above (no second sample)
        eff = learned_min_emissions(model, xf, nx, quantile=LENGTH_FLOOR_QUANTILE,
                                    base_floor=1, mults=mults)
        mp = model.map_estimate(xf, nx, **{**BEAM, "min_emissions": eff})
        # control: the same beam search with the >=1 floor lifted, so the empty tree
        # is reachable. min_emissions=0 and base_floor=0 together are what let a jet
        # whose truth is "no splitting" actually be answered correctly.
        n_map0.append(
            model.map_estimate(xf, nx, **{**BEAM, "min_emissions": 0}).multiplicity
            if MAP_ALLOW_EMPTY else -1
        )
        # MBR: the drawn tree of least expected perturbative-Lund EMD to the posterior
        mbr = model.map_or_mbr(xf, nx, draws=draws,
                               **{**DECODE, "point_estimator": "mbr",
                                  "mbr_backend": MBR_BACKEND,
                                  "mbr_n_candidates": MBR_N_CANDIDATES})
        # posterior predictive: one draw per jet, coordinates sampled where possible
        pick = draws[int(rng.integers(len(draws)))] if draws else []
        pv = draw_coords(model, xf, nx, pick)
        kap = ar_kappa(model, xf, nx, pick)

        raw["truth"].append(np.asarray(item["yraw"].numpy(), dtype=float))
        raw["rsd"].append(np.asarray(node_raw(*jets[i]["x"]), dtype=float))
        raw["map"].append(pe_coords(mp))
        raw["mbr"].append(pe_coords(mbr))
        raw["post"].append(pv)
        wjet.append(W_ALL[i])
        kappas.append(kap)
        risks.append(mbr.risk if mbr.risk is not None else np.nan)
    return (raw, np.array(wjet), kappas, np.array(risks, dtype=float),
            np.array(n_map0, dtype=int))

### 4a. Cost probe

MBR dominates the runtime: `MBR_N_CANDIDATES x K_DRAWS` optimal-transport solves per jet.
Measure on a handful of jets before committing to `N_JETS`.

In [ ]:
_probe_n = min(20, len(ds))
_t0 = time.perf_counter()
eval_jets(range(_probe_n))
_dt = (time.perf_counter() - _t0) / max(_probe_n, 1)

print(f"{_dt * 1e3:7.1f} ms / jet   (K={K_DRAWS}, MBR backend={MBR_BACKEND!r}, "
      f"candidates={MBR_N_CANDIDATES or 'all'})")
print(f"-> N_JETS={N_JETS} would take about {_dt * N_JETS / 60:.1f} min")
if MBR_BACKEND == "pot":
    print("   too slow? MBR_BACKEND='energyflow' is the SAME metric ~6x faster (identical")
    print("   winner on 99.3% of jets), so it costs nothing but the dependency. 'surrogate'")
    print("   is faster still but a different risk function -- iterate with it, report 'pot'")
    print("   or 'energyflow'. Lowering MBR_N_CANDIDATES / K_DRAWS also works.")

### 4b. Run it

In [ ]:
N = min(N_JETS, len(ds))
_t0 = time.perf_counter()
raw, w_jet, kappas, mbr_risk, n_map_unfloored = eval_jets(range(N))
print(f"evaluated {N} jets in {(time.perf_counter() - _t0) / 60:.1f} min "
      f"(K={K_DRAWS} draws each)")


def pack(arrays, weights):
    """Flatten per-jet (n,4) arrays into one splitting-level table.

    Every splitting carries its jet's weight (the convention in
    lund_rntuple_histograms.ipynb's `pooled`), its splitting index t, and its jet id --
    so slicing by splitting index downstream is a boolean mask, not a re-loop.
    """
    keep = [(i, a, w) for i, (a, w) in enumerate(zip(arrays, weights)) if len(a)]
    if not keep:
        return {"v": np.zeros((0, 4)), "w": np.zeros(0), "t": np.zeros(0, int),
                "jet": np.zeros(0, int)}
    return {
        "v": np.concatenate([a for _, a, _ in keep]),
        "w": np.concatenate([np.full(len(a), w, dtype=float) for _, a, w in keep]),
        "t": np.concatenate([np.arange(len(a)) for _, a, _ in keep]),
        # index into the EVALUATED set (not into `keep`), so the same id means the
        # same jet in every series even though different series drop different jets
        "jet": np.concatenate([np.full(len(a), i, dtype=int) for i, a, _ in keep]),
    }


POOL = {s: pack(raw[s], w_jet) for s in SERIES}          # splitting level
NSPL = {s: np.array([len(a) for a in raw[s]]) for s in SERIES}   # per-jet multiplicity
KAPPA = np.concatenate(kappas) if any(len(k) for k in kappas) else np.zeros(0)

print()
print(f"{'series':<8}{'splittings':>12}{'mean mult':>12}{'mean ln kt':>12}")
for s in SERIES:
    v = POOL[s]["v"]
    mk = float(np.average(v[:, 1], weights=POOL[s]["w"])) if len(v) else float("nan")
    print(f"{s:<8}{len(v):>12,}{NSPL[s].mean():>12.2f}{mk:>12.3f}")
if np.isfinite(mbr_risk).any():
    print(f"\nmean MBR risk (expected Lund-EMD to the posterior): "
          f"{np.nanmean(mbr_risk):.4f}")

## 5a. The empty tree — the observable v1 could not see

On this population $17\%$ of jets have **no** parton-level primary splitting: hadronisation
manufactured every splitting you see at hadron level, and the correct prediction is
*nothing*. v1 deleted these jets, so it could not ask whether the model knows that.

Two things get measured here. **Does the model reproduce the rate** $P(n{=}0)$ — a
population question, and the sharpest single test of whether it has learned when
hadronisation invents structure. And **does the estimator get those jets right**, one at a
time — which turns out to be a property of the *decode*, not of the model.

There are two independent reasons a MAP can miss them, and the `MAP_ALLOW_EMPTY` control
row separates them:

- **The floor.** `decode.min_emissions = 1` makes the empty tree unreachable outright.
  Lifting it is what the control row does.
- **The mode.** With a multiplicity head, `map_estimate` returns
  $\hat n = \arg\max_n q(n \mid x)$. Even a model that puts real mass at $n{=}0$ will
  almost never put its *peak* there, because that mass has to beat $n{=}1$ and $n{=}2$
  outright — so the MAP is non-empty whatever the floor says.

The second usually dominates, and it is the same mode-vs-distribution effect as everywhere
else in this notebook, one level up: it is the *length* that is being moded. On the
walkthrough `ar_junipr_v3` checkpoint the model carries genuine information about which
jets are empty — mean $q(0 \mid x) = 0.16$ on truth-empty jets against $0.08$ on the rest,
AUC $0.77$ — and the MAP discards all of it, with or without the floor. So if the control
row and the floored row agree, the floor is not your problem; the argmax is.

**MBR does not rescue this, and the reason is worth knowing.** It is mode-free and
floor-free, so it *could* return the empty tree — but under the default `pot` metric it
essentially never does (~0.2%). The perturbative-Lund EMD carries an imbalance penalty
(`mbr_R`, the Lund-plane diameter) for unmatched weight, and an empty cloud is nothing but
unmatched weight, so its risk against every non-empty draw is near-maximal. The empty tree
is not merely unlikely under this risk, it is close to the worst possible answer.

That makes the empty-tree column **backend-dependent in a way the other panels are not**.
On identical draws: `pot` and `energyflow` both give $P(\hat n = 0) \approx 0.2\%$ and
recover $0\%$ of the truth-empty jets, while `surrogate` gives $57\%$ and $82\%$ — because
a normalised binned-image $\chi^2$ does not punish an empty image the same way. Neither is
"right": they are different risk functions, and this observable is where they diverge
hardest. Read this section together with `MBR_BACKEND`, printed below, and do not carry a
conclusion from one backend to the other.

The net for the default decode is that **no point estimator can say "nothing"** — MAP
because of the argmax, MBR because of the imbalance penalty — while the posterior does.
Recovering those jets needs a decision rule that can express emptiness, e.g. thresholding
$q(0 \mid x)$ directly, not a different floor.

In [ ]:
IS0 = {s: (NSPL[s] == 0) for s in SERIES}
t0 = IS0["truth"]
wsum = w_jet.sum()

P_TRUE0 = float(w_jet[t0].sum() / wsum)

# q(N=0|x) -- the belief itself, before any decode touches it. Computed UP FRONT because
# the gate below is nothing but a threshold on it, so the two belong in one table.
q0 = np.array([float(model.length_pmf(ds[i]["xf"].unsqueeze(0).to(device),
                                      torch.tensor([ds[i]["nx"]], device=device))[0])
               for i in range(N)])
TAU = (empty_threshold_for_rate([np.array([v, 1.0 - v]) for v in q0], P_TRUE0)
       if EMPTY_THRESHOLD is None else float(EMPTY_THRESHOLD))
GATED = (q0 >= TAU) if TAU > 0 else np.zeros(N, dtype=bool)


def p0(mask):
    """Weighted P(n=0) for a per-jet boolean mask."""
    return float(w_jet[mask].sum() / wsum)


def recall0(mask):
    """P(n_hat=0) on the truth-empty jets. None with no truth-empty jets in the sample
    (REQUIRE_TRUTH_SPLITTING=True makes that exact): dividing by an empty class is not a
    4e11x overprediction, it is no question. The printed table and the record below both
    go through here, so the JSON cannot drift from the numbers you read."""
    return float(w_jet[mask & t0].sum() / w_jet[t0].sum()) if t0.any() else None


def zero_row(label, mask):
    """P(n=0), its ratio to truth's, and the hit rate on the truth-empty jets."""
    p, r = p0(mask), recall0(mask)
    ratio = f"{p / P_TRUE0:>10.2f}x" if P_TRUE0 > 0 else f"{'--':>11}"
    cond = f"{100 * r:>33.1f}%" if r is not None else f"{'n/a':>34}"
    print(f"{label:<15}{100 * p:>9.2f}%{ratio}   {cond}")


print(f"{'series':<15}{'P(n=0)':>10}{'vs truth':>11}   "
      f"{'on truth-empty jets: P(n_hat=0)':>34}")
print("-" * 73)
print(f"{'truth':<15}{100 * P_TRUE0:>9.2f}%")
for s in SERIES:
    if s != "truth":
        zero_row(s, IS0[s])
if MAP_ALLOW_EMPTY and (n_map_unfloored >= 0).all():
    zero_row("map (no floor)", n_map_unfloored == 0)
if TAU > 0:
    zero_row(f"gate t={TAU:.3f}", GATED)

print()
print(f"truth is the empty tree for {100 * float(w_jet[t0].sum() / wsum):.1f}% of jets.")
print(f"decode.min_emissions = {DECODE['min_emissions']} forbids the MAP from returning one.")
if not t0.any():
    print("No truth-empty jets in this sample, so there is nothing here to score --")
    print("that is REQUIRE_TRUTH_SPLITTING=True removing them BY SELECTION, which is")
    print("exactly the blind spot v2 exists to close. Set it to False.")
elif MAP_ALLOW_EMPTY and (n_map_unfloored >= 0).all():
    if (n_map_unfloored == 0).sum() == 0:
        print("Lifting the floor changes NOTHING -- so the floor was not the binding")
        print("constraint. With a multiplicity head the MAP is argmax q(n|x), and the peak")
        print("lands at n=0 essentially never however much mass sits there. That is a mode")
        print("artifact, not a floor artifact, and only a different point estimator fixes it.")
    else:
        _cf = float(w_jet[(n_map_unfloored == 0) & t0].sum() / w_jet[t0].sum())
        print(f"Lifting the floor recovers {100 * _cf:.1f}% of the truth-empty jets, so here")
        print("the floor IS costing real accuracy and the fix belongs in decode.")

# How much does the model actually KNOW about emptiness, independent of any decode?
_auc_q0 = None                                   # one class empty -> nothing to rank
if t0.any() and (~t0).any():
    _r = np.argsort(np.argsort(q0)) + 1          # AUC by rank statistic, no sklearn
    _auc_q0 = (_r[t0].mean() - (t0.sum() + 1) / 2) / (~t0).sum()
    print("\nq(N=0|x), the belief itself, before any decode touches it:")
    print(f"  mean on truth-empty jets    {q0[t0].mean():.4f}")
    print(f"  mean on truth-nonempty jets {q0[~t0].mean():.4f}")
    print(f"  AUC {_auc_q0:.3f}  -> the information is there; the argmax is what throws it away.")
    print(f"  mean q(0|x) = {q0.mean():.4f} against a true rate of {P_TRUE0:.4f}:")
    print(f"  the head is UNDER-CONFIDENT by {P_TRUE0 / max(q0.mean(), 1e-9):.2f}x.")
    print("  SBC/PIT do not catch this -- they rank against the sampler's own draws, so")
    print("  a uniformly squashed q(N|x) still passes. The gate works anyway because it")
    print("  thresholds the RANKING, which a monotone squash leaves alone.")
    if TAU > 0:
        _tp = float(w_jet[GATED & t0].sum())
        print(f"\ndecode.empty_threshold = {TAU:.4f} "
              f"({100 * float((q0 < TAU).mean()):.1f}th pct of q(0|x))")
        if EMPTY_THRESHOLD is None:
            print("  RATE-MATCHED ON THIS SAMPLE, so the P(n=0) column above reproduces")
            print("  truth by construction. Fit tau on held-out jets before quoting it.")
        print(f"  precision {_tp / max(float(w_jet[GATED].sum()), 1e-9):.3f}"
              f"   recall {_tp / max(float(w_jet[t0].sum()), 1e-9):.3f}"
              "   -- the population is right, the per-jet call is not solved.")
        print("  Unlike every other number in this section it is BACKEND-INDEPENDENT:")
        print("  the gate never touches the MBR risk.")
if not REQUIRE_TRUTH_SPLITTING and not t0.any():
    print("\n(no truth-empty jets in this sample -- raise N_JETS)")


# Handed to section 13, which is the only thing that outlives the kernel. The metrics
# JSON carried `selection.p_truth_empty` -- the TRUTH rate alone -- so every predicted
# row above, the floor control, the gate and the AUC existed only as cell output, and
# cell output does not survive a restart or a `git pull` (nbstripout smudges the
# notebook back from a stripped blob). None, not 0.0, wherever the sample cannot answer.
EMPTY_TREE = {
    "p_truth": P_TRUE0,
    "n_truth_empty": int(t0.sum()),
    "min_emissions": int(DECODE["min_emissions"]),
    "map_allow_empty": bool(MAP_ALLOW_EMPTY),
    # The MBR row, alone among these, moves with the risk function -- the imbalance
    # penalty is what makes the empty cloud near-maximal risk -- so the backend has to
    # travel with the numbers or they cannot be compared across runs.
    "mbr_backend": MBR_BACKEND,
    "p_pred": {s: p0(IS0[s]) for s in SERIES if s != "truth"},
    "recall": {s: recall0(IS0[s]) for s in SERIES if s != "truth"},
    "map_no_floor": ({"p_pred": p0(n_map_unfloored == 0),
                      "recall": recall0(n_map_unfloored == 0)}
                     if MAP_ALLOW_EMPTY and (n_map_unfloored >= 0).all() else None),
    "gate": ({"tau": float(TAU),
              "rate_matched_on_this_sample": EMPTY_THRESHOLD is None,
              "p_pred": p0(GATED),
              "recall": recall0(GATED),
              "precision": (float(w_jet[GATED & t0].sum() / w_jet[GATED].sum())
                            if w_jet[GATED].sum() > 0 else None)}
             if TAU > 0 else None),
    "q0": {"mean": float(q0.mean()),
           "mean_truth_empty": float(q0[t0].mean()) if t0.any() else None,
           "mean_truth_nonempty": float(q0[~t0].mean()) if (~t0).any() else None,
           "auc": _auc_q0,
           "underconfidence": (float(P_TRUE0 / max(q0.mean(), 1e-9))
                               if P_TRUE0 > 0 else None)},
}


## 5. Support and validity — what the model can and cannot produce

Before any distance is quoted, three numbers decide how much of it is even the model's
fault.

**Out of window.** `Geometry.to_cell` *clips* rather than drops, so truth splittings outside
$(\ln 1/\Delta R,\ \ln k_t) \in$ the geometry ranges were piled into edge cells during
training, and the model can never emit outside them at all. Whatever fraction of truth lies
outside is an irreducible floor on every distance — so the distances in §11 are computed on
the **fiducial window** (the geometry's own ranges) and this fraction is reported beside
them rather than silently absorbed.

**Soft-drop violations.** The groomer enforces $z > z_\mathrm{cut}(\Delta R/R_0)^\beta$
([`lund_io.hpp`](../cpp/include/lund_io.hpp)), so truth and plain RSD violate it exactly
zero times. The coordinate head models $\ln z$ with an *unbounded* $\mathcal N(\mu,\sigma)$
and has no idea the boundary exists — a non-zero number in that column is a real,
quantified physics failure, not a plotting artefact.

**$k_t$-floor violations.** Likewise $k_t \ge$ `kt_floor` (1 GeV, hence $\ln k_t \ge 0$,
hence the geometry starting at 0). Point estimates cannot violate this; sampled coordinates
can, since the within-cell offset is bounded but the edge cell straddles nothing below 0.

In [ ]:
U_LO, U_HI = geom.ln_invdelta_range
V_LO, V_HI = geom.ln_kt_range


def support_row(v, w):
    if not len(v):
        return dict(out_of_window=np.nan, sd_violation=np.nan, ktfloor_violation=np.nan)
    tot = w.sum()
    oow = ((v[:, 0] < U_LO) | (v[:, 0] > U_HI) | (v[:, 1] < V_LO) | (v[:, 1] > V_HI))
    kt = v[:, 1] < V_LO
    if SD_KNOWN:
        sd = v[:, 2] <= (math.log(Z_CUT) - BETA * v[:, 0])
        sd_f = float(w[sd].sum() / tot)
    else:
        sd_f = float("nan")
    return dict(out_of_window=float(w[oow].sum() / tot),
                sd_violation=sd_f,
                ktfloor_violation=float(w[kt].sum() / tot))


SUPPORT = {s: support_row(POOL[s]["v"], POOL[s]["w"]) for s in SERIES}

print(f"{'series':<8}{'out of window':>15}{'soft-drop viol.':>17}{'kt-floor viol.':>16}")
print("-" * 56)
for s in SERIES:
    r = SUPPORT[s]
    def f(x):
        return "     n/a" if not np.isfinite(x) else f"{100 * x:7.3f}%"
    print(f"{s:<8}{f(r['out_of_window']):>15}{f(r['sd_violation']):>17}"
          f"{f(r['ktfloor_violation']):>16}")
print()
print(f"fiducial window: ln(1/dR) in [{U_LO}, {U_HI}],  ln kt in [{V_LO}, {V_HI}]"
      + (f",  ln z > {LNZ_FLOOR:.3f}" if SD_KNOWN else ""))
print("truth/RSD 'out of window' is the irreducible floor on every distance in section 11.")

## 6. The primary Lund plane

$\rho(\ln 1/\Delta R,\ \ln k_t)$ — weighted splittings per jet per unit area, on the model's
own $(0,6)^2$ window at `PLANE_NB` bins per axis (a multiple of `geometry.n_bins`, so the
model's cell granularity shows rather than hides).

**What a correct $\rho$ looks like.** At fixed coupling the primary Lund density is
approximately *flat*, $\rho \approx 2\alpha_s C_F/\pi$ — that plateau is the whole point of
the coordinates. The running coupling tilts it upward toward small $k_t$, and the
soft-drop and $k_t$-floor conditions cut hard edges into it. Structure the eye can see here
beats any scalar in §11: a prediction that reproduces the plateau *and* the edges is doing
QCD, one that reproduces only the marginals may just be matching one-dimensional shapes.

**Expect the point-estimate panels to look striped.** The model emits a discrete cell, then
places the node at the coordinate head's mode, which is bounded to $\pm$`half_u`,
`half_v` = half a cell. So MAP and MBR can only ever populate a lattice at the geometry's
resolution, and the plane shows it. The posterior panel samples the offsets instead, which
is why it looks smooth — the striping is the *estimator's* discreteness, not a defect of
the density the model learned.

In [ ]:
def plane_edges(nb):
    return [np.linspace(U_LO, U_HI, nb + 1), np.linspace(V_LO, V_HI, nb + 1)]


def plane(s, e, nb):
    """Splittings per jet per unit Lund area, plus the raw counts behind it."""
    p = POOL[s]
    h = np.histogram2d(p["v"][:, 0], p["v"][:, 1], bins=e, weights=p["w"])[0]
    n = np.histogram2d(p["v"][:, 0], p["v"][:, 1], bins=e)[0]
    area = (U_HI - U_LO) * (V_HI - V_LO) / nb ** 2
    return h / (w_jet.sum() * area), n


PE = plane_edges(PLANE_NB)
PLANES, PCOUNT = {}, {}
for s in SERIES:
    PLANES[s], PCOUNT[s] = plane(s, PE, PLANE_NB)
# A single hot bin would flatten the whole ramp on a shared linear scale, so the top
# of the scale is a high percentile of the populated bins, not the maximum.
vmax = float(np.percentile(np.concatenate([P[P > 0] for P in PLANES.values()]), 99))

# View limits only -- the binning stays on the model's full (0,6)^2 window, but a
# 100 GeV sample populates a corner of it and plotting the whole square would shrink
# every structure to a smudge.
_hit = np.sum([P for P in PLANES.values()], axis=0) > 0
_iu, _iv = np.flatnonzero(_hit.any(axis=1)), np.flatnonzero(_hit.any(axis=0))
XLIM = (PE[0][max(_iu[0] - 1, 0)], PE[0][min(_iu[-1] + 2, PLANE_NB)])
YLIM = (PE[1][max(_iv[0] - 1, 0)], PE[1][min(_iv[-1] + 2, PLANE_NB)])

fig, axes = plt.subplots(1, 5, figsize=(19.0, 3.9), sharey=True)
for ax, s in zip(axes, SERIES):
    P = np.ma.masked_where(PLANES[s] <= 0, PLANES[s])
    im = ax.pcolormesh(PE[0], PE[1], P.T, cmap=CMAP, vmin=0.0, vmax=vmax,
                       shading="flat", rasterized=True)
    ax.set_title(STYLE[s][2])
    ax.set_xlabel(LABEL["lnInvDelta"])
    ax.set_xlim(*XLIM)
    ax.set_ylim(*YLIM)
    ax.grid(False)
axes[0].set_ylabel(LABEL["lnkt"])
fig.colorbar(im, ax=axes, fraction=0.016, pad=0.012,
             label=r"$\rho$  [splittings / jet / unit area]")
fig.suptitle("Primary Lund plane density  (view cropped to the populated region; "
             f"binning is the full {geom.ln_invdelta_range} x {geom.ln_kt_range} window)",
             x=0.09, ha="left")
plt.show()

In [ ]:
RLO, RHI, N_MIN = 0.4, 2.5, 12
# The ratio map is rebinned to the GEOMETRY's own cells. At PLANE_NB=30 there is
# barely one splitting per bin, so a ratio there is pure Poisson noise painted in
# saturated colour; the model's cell grid is both the coarser binning the statistics
# support and the resolution at which the model actually makes decisions.
RE = plane_edges(geom.n_bins)
RPLANE, RCOUNT = {}, {}
for s in SERIES:
    RPLANE[s], RCOUNT[s] = plane(s, RE, geom.n_bins)


def plane_ratio(num, den, n_den, n_num):
    """Ratio map, gated on truth having enough entries to divide by.

    A bin where TRUTH is empty but the prediction is not is a real disagreement
    (ratio -> infinity), so it saturates the top of the scale rather than being
    blanked -- blanking would hide invented emissions.
    """
    out = np.full(num.shape, np.nan)
    ok = (den > 0) & (n_den >= N_MIN)
    out[ok] = num[ok] / den[ok]
    out[(den <= 0) & (n_num >= N_MIN)] = RHI
    return np.ma.masked_invalid(out)


fig, axes = plt.subplots(1, 4, figsize=(15.6, 3.9), sharey=True)
for ax, s in zip(axes, ("rsd", "map", "mbr", "post")):
    R = plane_ratio(RPLANE[s], RPLANE["truth"], RCOUNT["truth"], RCOUNT[s])
    im = ax.pcolormesh(RE[0], RE[1], R.T, cmap=DIV,
                       norm=mpl.colors.LogNorm(vmin=RLO, vmax=RHI),
                       shading="flat", rasterized=True)
    ax.set_title(f"{STYLE[s][2]}  /  truth")
    ax.set_xlabel(LABEL["lnInvDelta"])
    ax.set_xlim(*XLIM)
    ax.set_ylim(*YLIM)
    ax.grid(False)
axes[0].set_ylabel(LABEL["lnkt"])
fig.colorbar(im, ax=axes, fraction=0.016, pad=0.012, label="ratio to truth")
fig.suptitle(f"Where each estimate over- and under-populates the plane, on the model's "
             f"own {geom.n_bins}x{geom.n_bins} cells   (grey = agrees, blue = too few, "
             f"red = too many, blank = fewer than {N_MIN} truth splittings)",
             x=0.06, ha="left")
plt.show()

## 7. Coordinate marginals, pooled over all splittings and jets

Weighted, **unit-area normalised**, on the C++ app's edges coarsened by `REBIN` — so any
panel here overlays [`lund_rntuple_histograms.ipynb`](lund_rntuple_histograms.ipynb)
without rebinning. The strip under each panel is the ratio to truth.

> **These are densities, not counts — so they will look like better $x$-vs-$y$ agreement
> than the same observables in `lund_rntuple_histograms.ipynb`, which plots weighted
> counts.** Hadronisation changes the *rate* of primary splittings a lot and their *shape*
> very little: on `cpp/test_data/jets.root` the hadron level carries ~23% more splittings
> than the parton level, so in a raw-count plot the $x$ curve sits above $y$ in every bin
> and that offset is most of what the eye reads as disagreement. Normalising to unit area
> divides it out on purpose — it is what makes $W_1$, KS and $\chi^2$ *shape* distances.
> The rate has not gone missing: it is §10's $k_t$-cut spectrum and the rate table in §13.
> Read the two together, or the model will look better here than it is.

**Read $\psi$ with the $\kappa$ panel.** The point estimate's $\psi$ is
$\mu = \operatorname{atan2}(b, a)$ — the von Mises *mode*. As $\kappa \to 0$ the conditional
becomes uniform and its mode carries essentially no information, so the MAP/MBR $\psi$
marginal can be sharply structured even when the posterior is perfectly flat and correct.
The $\kappa$ histogram says how much to discount that panel; $\psi$ should be uniform on
$(-\pi, \pi)$ by azimuthal symmetry.

In [ ]:
def marginal_panel(ax, rax, key, series=SERIES, t_mask=None, title=""):
    e = edges(key)
    dens = {}
    for s in series:
        p = POOL[s]
        m = np.ones(len(p["v"]), dtype=bool) if t_mask is None else t_mask(p)
        c, err = h1_sumw2(p["v"][m, COL[key]], p["w"][m], e)
        dens[s] = density(c, err, e)[0]
    if "rsd" in dens:
        fill(ax, dens["rsd"], e, C_RSD_F, C_RSD_E, label=STYLE["rsd"][2])
    for s in series:
        if s == "rsd":
            continue
        col, ls, lab = STYLE[s]
        step(ax, dens[s], e, col, label=lab, lw=2.2 if s == "truth" else 1.7,
             ls=ls, z=5 if s == "truth" else 3)
        if s != "truth":
            step(rax, ratio_of(dens[s], dens["truth"]), e, col, lw=1.4, ls=ls)
    if "rsd" in dens:
        step(rax, ratio_of(dens["rsd"], dens["truth"]), e, C_RSD_E, lw=1.2)
    zoom(ax, list(dens.values()), e)
    rax.set_xlim(ax.get_xlim())
    finish(ax, title=title, ylabel="density")
    rax.set_xlabel(LABEL[key])
    return dens


KEYS = ["lnInvDelta", "lnkt", "lnz", "psi"]
fig, pairs = ratio_axes(4, 1, w=4.3, h=3.5)
for (ax, rax), key in zip(pairs[0], KEYS):
    # Only a family with NO coordinate density (ar_junipr_v1) leaves ln z / psi as
    # placeholders; every other family draws them through sample_coordinates. Plotting a
    # filler constant as a prediction would be a lie, so those series are dropped instead.
    ser = SERIES if (CONT or key in ("lnInvDelta", "lnkt")) else ("truth", "rsd")
    marginal_panel(ax, rax, key, series=ser,
                   title=LABEL[key] + ("" if CONT or key in ("lnInvDelta", "lnkt")
                                       else "   (no coordinate density)"))
if not CONT:
    for c in (2, 3):
        pairs[0][c][0].text(0.03, 0.97, "model series dropped: this family\nhas no "
                            "coordinate density (ln z, psi unset)", ha="left", va="top",
                            transform=pairs[0][c][0].transAxes, fontsize=7.5,
                            color=MUTED, bbox=dict(facecolor=SURFACE, edgecolor="none",
                                                   pad=2.0))
fig_legend(fig, pairs[0][0][0],
           "Groomed-observable marginals, pooled over every splitting and jet")
plt.show()

In [ ]:
if CONT and len(KAPPA):
    fig, ax = plt.subplots(figsize=(4.4, 2.9))
    ax.hist(KAPPA, bins=np.linspace(0, max(2.0, np.percentile(KAPPA, 99)), 40),
            color=C_POST, alpha=0.85)
    ax.axvline(1.0, color=MUTED, lw=1.0, ls="--")
    q = np.mean(KAPPA < 1.0)
    finish(ax, xlabel=r"von Mises $\kappa$ of the $\psi$ head",
           ylabel="splittings",
           title=rf"$\psi$ is informative only where $\kappa$ is large "
                 rf"({100 * q:.0f}% of splittings have $\kappa<1$)")
    plt.show()
    print(f"median kappa = {np.median(KAPPA):.3f}    "
          f"kappa < 1 for {100 * np.mean(KAPPA < 1.0):.1f}% of splittings")
    print("Low kappa => the psi MODE is close to arbitrary, so discount the psi panel")
    print("above for MAP/MBR. The posterior series samples the von Mises, so it stays honest.")
else:
    print("kappa diagnostic unavailable (no continuous coordinate head)")

## 8. Marginals by splitting index

The request's centre of gravity: *the best point estimate of the Lund variables for each
split*. Splitting $t=0$ is the hardest and best determined — every series should agree
there. Disagreement growing with $t$ is the expected signature (later splittings are softer
and the hadron-to-parton map is less constrained); disagreement already at $t=0$ is not.

In [ ]:
T_LABELS = [f"$t={t}$" for t in T_SLICES] + [f"$t\\geq{T_SLICES[-1] + 1}$"]
T_MASKS = [(lambda p, t=t: p["t"] == t) for t in T_SLICES] + \
          [(lambda p: p["t"] > T_SLICES[-1])]
KEYS_T = KEYS if CONT else ["lnInvDelta", "lnkt"]

fig, pairs = ratio_axes(len(T_MASKS), len(KEYS_T), w=3.5, h=3.1)
for row, key in zip(pairs, KEYS_T):
    for (ax, rax), tm, tl in zip(row, T_MASKS, T_LABELS):
        marginal_panel(ax, rax, key, t_mask=tm, title=f"{LABEL[key]}   {tl}")
fig_legend(fig, pairs[0][0][0], "Coordinate marginals split by splitting index",
           top=0.90 if len(KEYS_T) > 2 else 0.84)
plt.show()

## 9. Ladder profiles

Mean $\ln 1/\Delta R$ and mean $\ln k_t$ as a function of splitting index, band = weighted
16–84 percentile. QCD's angular ordering says the declustering sequence must march inward:
$\ln 1/\Delta R$ rising with $t$, $\ln k_t$ falling. A prediction with a **flat** ladder has
learned the pooled marginals without learning the sequence, which the pooled panels in §7
cannot detect.

The third panel is the survival curve — the fraction of jets still emitting at depth $t$,
i.e. the multiplicity distribution read cumulatively.

In [ ]:
def profile(s, col):
    mu, lo, hi = [], [], []
    p = POOL[s]
    for t in range(T_LADDER):
        m = p["t"] == t
        if m.sum() < 3:
            mu.append(np.nan); lo.append(np.nan); hi.append(np.nan); continue
        v, w = p["v"][m, col], p["w"][m]
        o = np.argsort(v)
        cw = np.cumsum(w[o]) / w[o].sum()
        mu.append(float(np.average(v, weights=w)))
        lo.append(float(v[o][np.searchsorted(cw, 0.16)]))
        hi.append(float(v[o][np.searchsorted(cw, 0.84)]))
    return np.array(mu), np.array(lo), np.array(hi)


ts = np.arange(T_LADDER)
fig, axes = plt.subplots(1, 3, figsize=(14.4, 3.6))
for ax, (col, key) in zip(axes[:2], [(0, "lnInvDelta"), (1, "lnkt")]):
    for s in SERIES:
        c, ls, lab = STYLE[s]
        mu, lo, hi = profile(s, col)
        ax.fill_between(ts, lo, hi, color=c, alpha=0.10, lw=0, zorder=1)
        ax.plot(ts, mu, color=c, ls=ls, marker="o", ms=3.4,
                lw=2.2 if s == "truth" else 1.7, label=lab, zorder=4)
    finish(ax, xlabel="splitting index $t$", ylabel="weighted mean " + LABEL[key],
           title=LABEL[key] + " ladder")

for s in SERIES:
    c, ls, lab = STYLE[s]
    surv = [float(w_jet[NSPL[s] > t].sum() / w_jet.sum()) for t in ts]
    axes[2].plot(ts, surv, color=c, ls=ls, marker="o", ms=3.4,
                 lw=2.2 if s == "truth" else 1.7, label=lab)
axes[2].set_yscale("log")
finish(axes[2], xlabel="splitting index $t$", ylabel="fraction of jets reaching $t$",
       title="survival")
# Trim to the depths any series actually reaches -- T_LADDER is an upper bound, not
# a promise that the sample is that deep.
t_max = max((int(NSPL[s].max()) for s in SERIES), default=1)
for ax in axes:
    ax.set_xlim(-0.3, min(T_LADDER - 1, t_max) + 0.3)
fig.tight_layout()
fig_legend(fig, axes[0], "Does the predicted sequence march inward the way QCD demands?",
           top=0.78)
plt.show()

## 10. $k_t$-cut multiplicity spectrum

$N(\ln k_t > c)$ per jet, weight-averaged, as the cut sweeps the geometry's $\ln k_t$ range.

This is the panel to keep if only one survives. Multiplicity alone says *how many*
emissions; the marginals say *where* emissions sit; this says both at once — a model that
gets the count right by putting emissions at the wrong hardness shows up here as a curve of
the right endpoint but the wrong slope.

In [ ]:
CUTS = np.linspace(V_LO, V_HI, 25)


def n_above(s):
    p, out = POOL[s], []
    wsum = w_jet.sum()
    for c in CUTS:
        m = p["v"][:, 1] > c
        out.append(float(p["w"][m].sum() / wsum))
    return np.array(out)


NCUT = {s: n_above(s) for s in SERIES}

fig, axes = plt.subplots(2, 1, figsize=(5.6, 4.6), sharex=True,
                         gridspec_kw={"height_ratios": [3, 1], "hspace": 0.10})
for s in SERIES:
    c, ls, lab = STYLE[s]
    axes[0].plot(CUTS, NCUT[s], color=c, ls=ls, lw=2.2 if s == "truth" else 1.7, label=lab)
    if s != "truth":
        axes[1].plot(CUTS, ratio_of(NCUT[s], NCUT["truth"]), color=c, ls=ls, lw=1.4)
axes[1].axhline(1.0, color=MUTED, lw=0.8)
axes[1].set_ylim(0.0, 2.0)
axes[0].set_yscale("log")
axes[0].tick_params(labelbottom=False)
finish(axes[0], ylabel=r"$\langle N(\ln k_t > c)\rangle$ per jet",
       title=r"$k_t$-cut multiplicity spectrum", legend=True)
finish(axes[1], xlabel=r"cut $c$ on $\ln(k_t/\mathrm{GeV})$", ylabel="/ truth")
plt.show()

## 11. Leading emission and multiplicity

The hardest-$k_t$ splitting of each jet, and the multiplicity distribution.

`eval/closure.leading_emission_cell` argmaxes over *cell ids*; this argmaxes the continuous
$\ln k_t$ column — the same observable at full precision. The numbers here will therefore
not equal `eval_metrics.json`'s, by construction rather than by disagreement.

> **These leading-emission panels are conditional, and in v2 the condition differs per
> series.** A jet with no splittings has no leading emission, so each series contributes
> only its non-empty jets — and truth is empty ~17% of the time while a floored MAP never
> is. So this is $p(\text{leading} \mid n>0)$ per series, over different jets, *not* a
> like-for-like comparison. The counts are printed below; the multiplicity panel and §5a
> are where the $n{=}0$ population is accounted for. Equalising the subsets would mean
> conditioning on truth again, which is the whole thing v2 exists to avoid.

In [ ]:
def leading(s):
    """(n_jets, 4) coords of each jet's hardest-kt splitting, + its weight."""
    rows, ws = [], []
    for a, w in zip(raw[s], w_jet):
        if len(a):
            rows.append(a[int(np.argmax(a[:, 1]))])
            ws.append(w)
    return (np.array(rows) if rows else np.zeros((0, 4))), np.array(ws)


LEAD = {s: leading(s) for s in SERIES}
print("jets contributing a leading emission (the rest have no splitting at all):")
for s in SERIES:
    print(f"  {s:<8}{len(LEAD[s][0]):>7,} / {len(w_jet):,}"
          f"   ({100 * len(LEAD[s][0]) / max(len(w_jet), 1):.1f}%)")

lead_keys = ["lnkt", "lnInvDelta"] + (["lnz"] if CONT else [])
fig, pairs = ratio_axes(len(lead_keys) + 1, 1, w=4.2, h=3.4)
for (ax, rax), key in zip(pairs[0], lead_keys):
    e = edges(key)
    d = {}
    for s in SERIES:
        v, w = LEAD[s]
        d[s] = density(*h1_sumw2(v[:, COL[key]], w, e), e)[0]
    fill(ax, d["rsd"], e, C_RSD_F, C_RSD_E, label=STYLE["rsd"][2])
    for s in SERIES:
        if s == "rsd":
            continue
        c, ls, lab = STYLE[s]
        step(ax, d[s], e, c, label=lab, lw=2.2 if s == "truth" else 1.7, ls=ls)
        if s != "truth":
            step(rax, ratio_of(d[s], d["truth"]), e, c, lw=1.4, ls=ls)
    step(rax, ratio_of(d["rsd"], d["truth"]), e, C_RSD_E, lw=1.2)
    zoom(ax, list(d.values()), e)
    rax.set_xlim(ax.get_xlim())
    finish(ax, title="leading emission: " + LABEL[key], ylabel="density")
    rax.set_xlabel(LABEL[key])

ax, rax = pairs[0][-1]
e = edges("mult")
dm = {s: density(*h1_sumw2(NSPL[s], w_jet, e), e)[0] for s in SERIES}
fill(ax, dm["rsd"], e, C_RSD_F, C_RSD_E)
for s in SERIES:
    if s == "rsd":
        continue
    c, ls, _ = STYLE[s]
    step(ax, dm[s], e, c, lw=2.2 if s == "truth" else 1.7, ls=ls)
    if s != "truth":
        step(rax, ratio_of(dm[s], dm["truth"]), e, c, lw=1.4, ls=ls)
step(rax, ratio_of(dm["rsd"], dm["truth"]), e, C_RSD_E, lw=1.2)
zoom(ax, list(dm.values()), e)
rax.set_xlim(ax.get_xlim())
finish(ax, title="multiplicity", ylabel="density")
rax.set_xlabel(LABEL["mult"])
fig_legend(fig, pairs[0][0][0], "Leading emission and multiplicity")
plt.show()

## 12. Distribution distances

Three distances, deliberately redundant, so no single statistic drives the verdict:

- **$W_1$** — Wasserstein-1, the mean transport distance. Binning-free, carries the
  observable's units, sensitive to *shifts*. Weight-aware
  (`scipy.stats.wasserstein_distance` takes `u_weights` / `v_weights`).
- **KS** — the largest gap between the two CDFs. Binning-free, sensitive to *shape*
  disagreement anywhere. `scipy.stats.ks_2samp` is **not** weight-aware, so `ks_w` below
  computes the statistic directly off the weighted ECDFs rather than quietly dropping the
  jet weights.
- **$\chi^2/\mathrm{ndf}$** — bin-wise, with Sumw2 errors on both histograms. The familiar
  HEP number; the one that depends on the binning.

**$\psi$ is a circle, not a line**, so its rows (marked `(circ)`) use the rotation-invariant
twins: circular $W_1$, which lets the transport plan wrap, and **Kuiper's** $V = \sup(F_a -
F_b) + \sup(F_b - F_a)$ in place of KS. Applying the linear versions to an azimuth charges a
full $2\pi$ of transport for mass sitting either side of the branch cut, so two identical
distributions can score as maximally far apart. Kuiper's $V$ lives on a different scale from
KS ($[0,2]$ rather than $[0,1]$), which is harmless here because $\psi$ rows are only ever
compared against $\psi$ rows — the ratio is the quantity that carries across observables.

Each is evaluated pred-vs-truth and RSD-vs-truth, and the headline is the **improvement
ratio** $r = d(\hat y, y) / d(x, y)$. Plain RSD is the denominator, so its own ratio is
exactly 1.0 — a built-in check that the plumbing is wired to the right series.

Everything is computed on the **fiducial window** of §5, because scoring the model on truth
it structurally cannot reach measures the geometry, not the model. The out-of-window
fractions from §5 are the caveat that travels with these numbers.

### Not every row can be won

An improvement ratio divides by $d(x, y)$. When that baseline distance is itself at the
level of statistical noise, the ratio is not a hard test — it is **meaningless**, and every
estimator scores above 1 no matter how good it is.

That is not hypothetical here. Hadronisation barely changes the *shape* of some of these
observables: $\ln z$ is pinned between two hard walls only
$\ln(0.5/0.1) = 1.61$ nats apart, and $\psi$ is uniform by azimuthal symmetry at both
levels. There is nothing in those distributions for the model to recover, so plain RSD is
already right and cannot be beaten.

So each row carries a **noise floor**: the same three distances measured between two
independent bootstrap resamples of *truth against itself*, at that row's own sample size.
The bootstrap resamples **jets, not splittings** — splittings within a jet share a weight
and correlated kinematics, so resampling them independently would understate the floor.

A row is **scoreable** only when $d(x, y)$ exceeds its floor. Rows that fail are still
printed — with their distances and their floor, so you can see *why* — but marked `[n/s]`,
their ratio columns blanked, and they are excluded from the headline. Otherwise a model
that is right about everything still posts a headline above 1, purely from rows where there
was never anything to fix.

### Shape and rate are different questions

**All three distances are shape distances.** They run on unit-area histograms, so they are
blind to overall normalisation by construction — they ask "is this the right shape?", never "is this
the right *number* of emissions?". That is the axis §10's $k_t$-cut spectrum lives on, and a
model can win there while losing every row above (or the reverse). So the rate is scored
separately, on its own terms: emissions per jet above a cut, and the relative deviation from
truth, with the same lower-is-better / ratio-against-RSD convention.

In [ ]:
def fid_window(key):
    """Fiducial edges: the plotting edges restricted to the model's own support."""
    e = edges(key)
    if key == "lnInvDelta":
        lo, hi = U_LO, U_HI
    elif key == "lnkt":
        lo, hi = V_LO, V_HI
    elif key == "lnz":
        lo, hi = LNZ_FLOOR, 0.0
    else:
        lo, hi = e[0], e[-1]
    sub = e[(e >= lo - 1e-9) & (e <= hi + 1e-9)]
    return sub if sub.size >= 3 else np.linspace(lo, hi, 26)


def _wecdf(x, w, grid):
    """Weighted ECDF of (x, w) evaluated on `grid`."""
    o = np.argsort(np.asarray(x, float))
    xs = np.asarray(x, float)[o]
    cw = np.cumsum(np.asarray(w, float)[o])
    cw = cw / cw[-1]
    idx = np.searchsorted(xs, grid, side="right")
    return np.where(idx > 0, cw[np.clip(idx - 1, 0, None)], 0.0)


def w1(a, wa, b, wb):
    """Wasserstein-1. Weight-aware."""
    if len(a) < 2 or len(b) < 2:
        return float("nan")
    return float(wasserstein_distance(a, b, wa, wb))


def w1_circular(a, wa, b, wb, lo, hi, n=2048):
    """Wasserstein-1 on a CIRCLE -- the right distance for psi.

    Linear W1 measures the +pi/-pi wrap as a 2*pi transport, so two identical
    azimuthal distributions read as maximally far apart purely because of where
    the branch cut falls. On the circle the optimal plan is free to rotate:
        W1 = min_theta  Int |F_a(x) - F_b(x) - theta| dx
    (Delon, Salomon & Sobolevski 2010), and on a uniform grid the minimising
    theta is just the median of the CDF difference.
    """
    if len(a) < 2 or len(b) < 2:
        return float("nan")
    L = hi - lo
    g = lo + (np.arange(n) + 0.5) * L / n
    d = _wecdf(a, wa, g) - _wecdf(b, wb, g)
    return float(np.mean(np.abs(d - np.median(d))) * L)


def ks_w(a, wa, b, wb):
    """Two-sample KS statistic off the WEIGHTED ECDFs.

    scipy.stats.ks_2samp ignores weights; with per-jet ROOT weights in play that
    would silently answer a different question. sup|F_a - F_b| over the merged
    support is the same statistic, computed honestly.
    """
    if len(a) < 2 or len(b) < 2:
        return float("nan")
    grid = np.union1d(np.asarray(a, float), np.asarray(b, float))
    return float(np.max(np.abs(_wecdf(a, wa, grid) - _wecdf(b, wb, grid))))


def kuiper_w(a, wa, b, wb):
    """Kuiper's V = sup(F_a - F_b) + sup(F_b - F_a) -- the circular KS.

    KS depends on where the circle was cut open; Kuiper's statistic is invariant
    under rotation, which is what an azimuth needs. Its scale differs from KS
    (V in [0,2]), so psi rows are only ever compared against psi rows.
    """
    if len(a) < 2 or len(b) < 2:
        return float("nan")
    grid = np.union1d(np.asarray(a, float), np.asarray(b, float))
    d = _wecdf(a, wa, grid) - _wecdf(b, wb, grid)
    return float(max(d.max(), 0.0) + max((-d).max(), 0.0))


def chi2_ndf(a, wa, b, wb, e, w2a=None, w2b=None):
    """Shape chi2/ndf on shared bins, unit-area normalised, Sumw2 errors.

    Bins where NEITHER distribution has an error are dropped, not counted -- an
    empty-empty bin is agreement about nothing and would deflate the statistic.
    ndf = (bins used) - 1; the one constraint is the shared normalisation.
    """
    pa, sa = density(*h1_sumw2(a, wa, e, w2a), e)
    pb, sb = density(*h1_sumw2(b, wb, e, w2b), e)
    var = sa ** 2 + sb ** 2
    use = var > 0
    ndf = int(use.sum()) - 1
    if ndf < 1:
        return float("nan")
    return float(np.sum((pa[use] - pb[use]) ** 2 / var[use]) / ndf)


MET = ("w1", "ks", "chi2")
RNG = np.random.default_rng(SEED + 1)


def dists(a, wa, b, wb, e, circular, w2a=None, w2b=None):
    if circular is None:
        d = {"w1": w1(a, wa, b, wb), "ks": ks_w(a, wa, b, wb)}
    else:
        d = {"w1": w1_circular(a, wa, b, wb, *circular), "ks": kuiper_w(a, wa, b, wb)}
    d["chi2"] = chi2_ndf(a, wa, b, wb, e, w2a, w2b)
    return d


def noise_floor(v, w, j, e, circular=None):
    """The distance you measure between TRUTH and ITSELF at this sample size.

    Two independent bootstrap resamples of truth, scored with the same three
    distances. Whatever comes back is pure sampling noise, so a plain-RSD distance
    at or below it carries no information -- and an improvement ratio that divides
    by it is meaningless, not merely imprecise.

    The bootstrap resamples JETS, not splittings: splittings within a jet share a
    weight and correlated kinematics, so a splitting-level resample would treat
    them as independent and understate the floor.

    Deliberately CONSERVATIVE in two ways, both of which can only hide a winnable
    row, never invent one: both resamples carry bootstrap noise where the real
    comparison uses the truth sample itself, and both are drawn at truth's size
    even when the other series has somewhat more splittings. This is a screening
    device for "is there anything here to win", not a p-value.
    """
    if len(v) < 8:
        return {m: float("nan") for m in MET}
    _, jc = np.unique(j, return_inverse=True)   # jet id -> 0..nb-1
    nb = int(jc.max()) + 1
    p = np.full(nb, 1.0 / nb)
    acc = {m: [] for m in MET}
    w2 = w ** 2
    for _ in range(N_BOOT):
        # A bootstrap resample IS a multinomial reweighting of the blocks, and W1/KS
        # are linear in the weights -- so the resample is a weight vector, not an
        # index shuffle. chi2 is the exception: a jet drawn c times stands for c
        # INDEPENDENT entries, contributing c*w**2 to Sumw2 and not (c*w)**2, so its
        # sum-of-squares term is passed explicitly. Folding it into the weight would
        # inflate the error bars, deflate the null chi2, and wave through rows that
        # are really inside the noise.
        ca, cb = RNG.multinomial(nb, p)[jc], RNG.multinomial(nb, p)[jc]
        wa, wb = w * ca, w * cb
        if wa.sum() <= 0 or wb.sum() <= 0:
            continue
        d = dists(v, wa, v, wb, e, circular, w2a=ca * w2, w2b=cb * w2)
        for m in MET:
            acc[m].append(d[m])
    return {m: (float(np.nanpercentile(acc[m], FLOOR_PCT))
                if len(acc[m]) and np.isfinite(acc[m]).any() else float("nan"))
            for m in MET}


def compare(vals, wts, jids, e, circular=None):
    """All three distances of every model series against truth, the ratios, and
    the noise floor that decides whether those ratios mean anything.

    `circular=(lo, hi)` switches W1 and KS to their rotation-invariant twins --
    used for psi, where the linear versions charge a full 2*pi for the branch cut.
    """
    out = {}
    ref_v, ref_w = vals["truth"], wts["truth"]
    base = {}
    for s in ("rsd",) + MODELS:
        if s not in vals or len(vals[s]) < 2:
            continue
        d = dists(vals[s], wts[s], ref_v, ref_w, e, circular)
        if s == "rsd":
            base = d
        out[s] = d
    for s, d in out.items():
        for m in MET:
            b = base.get(m, float("nan"))
            d[m + "_r"] = (d[m] / b) if (np.isfinite(b) and b > 0) else float("nan")
    floor = noise_floor(ref_v, ref_w, jids["truth"], e, circular)
    # scoreable == plain RSD is measurably further from truth than truth is from
    # itself. Where it is not, hadronisation left no shape difference to recover
    # and no estimator can post a ratio below 1.
    ok = {m: bool(np.isfinite(floor[m]) and np.isfinite(base.get(m, float("nan")))
                  and base[m] > floor[m]) for m in MET}
    return {"series": out, "floor": floor, "scoreable": ok}


def obs_splitting(key, t_mask=None):
    """A splitting-level observable, clipped to its fiducial window."""
    e = fid_window(key)
    lo, hi = e[0], e[-1]
    vals, wts, jids = {}, {}, {}
    for s in SERIES:
        p = POOL[s]
        m = np.ones(len(p["v"]), bool) if t_mask is None else t_mask(p)
        x = p["v"][m, COL[key]]
        keep = (x >= lo) & (x <= hi)
        vals[s], wts[s] = x[keep], p["w"][m][keep]
        jids[s] = p["jet"][m][keep]      # bootstrap blocks
    return vals, wts, jids, e


def obs_jet(getter, e):
    """A per-jet observable (multiplicity, leading emission). One entry per jet,
    so every bootstrap block is a singleton and the block bootstrap reduces to
    the ordinary one -- which is correct here."""
    vals, wts, jids = {}, {}, {}
    for s in SERIES:
        v, w = getter(s)
        vals[s], wts[s], jids[s] = v, w, np.arange(len(v))
    return vals, wts, jids, e


ROWS = []
CIRC = {"psi": (-np.pi, np.pi)}   # observables that live on a circle, not a line


def add(name, vals, wts, jids, e, key=None):
    ROWS.append((name, compare(vals, wts, jids, e, circular=CIRC.get(key))))


for key in (KEYS if CONT else ["lnInvDelta", "lnkt"]):
    mark = " (circ)" if key in CIRC else ""
    add(f"{key} (pooled){mark}", *obs_splitting(key), key=key)
    for t, tl in zip(T_SLICES, T_LABELS):
        add(f"{key} t={t}{mark}", *obs_splitting(key, (lambda p, t=t: p["t"] == t)), key=key)
    add(f"{key} t>={T_SLICES[-1] + 1}{mark}",
        *obs_splitting(key, lambda p: p["t"] > T_SLICES[-1]), key=key)

# per-jet observables
add("multiplicity", *obs_jet(lambda s: (NSPL[s].astype(float), w_jet), edges("mult")))
for key in (["lnkt", "lnInvDelta"] + (["lnz"] if CONT else [])):
    add(f"leading {key}",
        *obs_jet(lambda s, k=key: (LEAD[s][0][:, COL[k]], LEAD[s][1]), fid_window(key)))

# Lund quadrants: plain RSD is already close to truth for HARD emissions, so a pooled
# number hides where the model actually earns its keep. cell_region is the same
# quadrant definition eval/calibration.py stratifies on.
def quad_mask(label):
    def f(p):
        cells = np.array([geom.to_cell(u, v) for u, v in p["v"][:, :2]], dtype=int)
        return np.array([cell_region(int(c), geom) == label for c in cells])
    return f


for lab in REGION_LABELS:
    add(f"lnkt [{lab}]", *obs_splitting("lnkt", quad_mask(lab)))

# --- rates, scored separately -----------------------------------------------
# W1 / KS / chi2 all run on UNIT-AREA histograms, so they are blind to overall
# normalisation by construction -- they ask "the right shape?", never "the right
# number of emissions?". The kt-cut spectrum in section 10 is exactly the missing
# axis, so it is scored on its own terms: emissions per jet above a cut, and the
# relative deviation from truth. Same convention as everywhere else -- lower is
# better, and the ratio is against plain RSD.
n_jets_ev = len(w_jet)


def rate_per_jet(s, c, mult=None):
    """Emissions per jet above a kt cut. `mult` is a per-jet repeat count, so a
    bootstrap draw can weight a jet by how many times it was drawn -- a set-membership
    test would silently deduplicate the resample and bias the rate down by ~37%."""
    p = POOL[s]
    m = p["v"][:, 1] > c
    if mult is None:
        num, den = p["w"][m].sum(), w_jet.sum()
    else:
        num = (p["w"][m] * mult[p["jet"][m]]).sum()
        den = (w_jet * mult).sum()
    return float(num / den) if den > 0 else float("nan")


RATE_ROWS = []
for c in (0.0, 1.0, 2.0, 3.0):
    per_jet = {s: rate_per_jet(s, c) for s in SERIES}
    ref = per_jet["truth"]
    # Same gate as the shape rows: how far does truth's own rate move under a
    # jet-level resample? A baseline deviation inside that is not a deviation.
    boot = [rate_per_jet("truth", c,
                         np.bincount(RNG.integers(0, n_jets_ev, n_jets_ev),
                                     minlength=n_jets_ev).astype(float))
            for _ in range(N_BOOT)]
    floor = (float(np.nanpercentile(np.abs(np.array(boot) / ref - 1.0), FLOOR_PCT))
             if ref > 0 else float("nan"))
    base = abs(per_jet["rsd"] / ref - 1.0) if ref > 0 else float("nan")
    ok = bool(np.isfinite(floor) and np.isfinite(base) and base > floor)
    d = {}
    for s in ("rsd",) + MODELS:
        dev = abs(per_jet[s] / ref - 1.0) if ref > 0 else float("nan")
        d[s] = {"n_per_jet": per_jet[s], "rel_dev": dev,
                "rel_dev_r": (dev / base) if (np.isfinite(base) and base > 0)
                else float("nan")}
    RATE_ROWS.append((f"N(ln kt > {c:g})", ref, floor, ok, d))

_ns = {m: sum(1 for _, d in ROWS if d["scoreable"][m]) for m in MET}
print(f"{len(ROWS)} shape observables + {len(RATE_ROWS)} rate observables "
      f"compared against truth")
print("scoreable (plain RSD measurably worse than truth-vs-truth noise): "
      + "  ".join(f"{m} {_ns[m]}/{len(ROWS)}" for m in MET)
      + f"   |  rate {sum(1 for r in RATE_ROWS if r[3])}/{len(RATE_ROWS)}")

## 13. Summary

The headline first: the geometric mean of the improvement ratio across every **scoreable**
observable (geometric, because ratios compose multiplicatively and one 10x outlier should
not dominate), and how many of them each estimator actually beat plain RSD on. Rows where
plain RSD is already inside the truth-vs-truth noise floor are excluded — see §12; the
`wins` denominators tell you how many rows survived.

Then one table per distance, and finally the **rate** table — which is the one to read
alongside §10, and the one that can disagree with the other three, since they are
normalisation-blind and it is nothing but normalisation.

**Lower is better everywhere**; the `rsd` column is the baseline the ratios divide by.

In [ ]:
def gmean(xs):
    xs = np.array([x for x in xs if np.isfinite(x) and x > 0], dtype=float)
    return float(np.exp(np.mean(np.log(xs)))) if xs.size else float("nan")


def md_table(header, rows):
    w = [max(len(str(header[i])), *(len(str(r[i])) for r in rows)) for i in range(len(header))]
    out = ["| " + " | ".join(str(h).ljust(w[i]) for i, h in enumerate(header)) + " |",
           "|" + "|".join("-" * (w[i] + 2) for i in range(len(header))) + "|"]
    for r in rows:
        out.append("| " + " | ".join(str(c).ljust(w[i]) for i, c in enumerate(r)) + " |")
    return "\n".join(out)


def fmt(x, p=4):
    return "--" if not np.isfinite(x) else f"{x:.{p}g}"


METRICS = [("w1", "W1"), ("ks", "KS"), ("chi2", "chi2/ndf")]


def scored(s, m):
    """Ratios from SCOREABLE rows only -- the rest divide by noise."""
    return [d["series"][s][m + "_r"] for _, d in ROWS
            if d["scoreable"][m] and s in d["series"]]


head_rows = []
for s in MODELS:
    r = [STYLE[s][2].replace("$", "")]
    for m, _ in METRICS:
        ratios = scored(s, m)
        won = sum(1 for x in ratios if np.isfinite(x) and x < 1.0)
        tot = sum(1 for x in ratios if np.isfinite(x))
        r += [fmt(gmean(ratios), 3), f"{won}/{tot}"]
    rate_r = [dd[s]["rel_dev_r"] for _, _, _, ok, dd in RATE_ROWS if ok and s in dd]
    r += [fmt(gmean(rate_r), 3),
          f"{sum(1 for x in rate_r if np.isfinite(x) and x < 1.0)}/"
          f"{sum(1 for x in rate_r if np.isfinite(x))}"]
    head_rows.append(r)

HEADLINE = md_table(
    ["estimator", "W1 ratio", "W1 wins", "KS ratio", "KS wins",
     "chi2 ratio", "chi2 wins", "rate ratio", "rate wins"], head_rows)

TABLES = {}
for m, mlab in METRICS:
    rows = []
    for name, d in ROWS:
        ser, ok = d["series"], d["scoreable"][m]
        cells = [name if ok else name + "  [n/s]"]
        cells += [fmt(ser[s][m]) if s in ser else "--" for s in ("rsd",) + MODELS]
        cells += [fmt(d["floor"][m])]
        cells += [(fmt(ser[s][m + "_r"], 3) if s in ser else "--") if ok else "n/s"
                  for s in MODELS]
        rows.append(cells)
    TABLES[m] = md_table(
        ["observable", "rsd", "map", "mbr", "post", "noise floor",
         "map/rsd", "mbr/rsd", "post/rsd"], rows)

rate_rows = []
for name, ref, floor, ok, d in RATE_ROWS:
    rate_rows.append(
        [name if ok else name + "  [n/s]", fmt(ref, 3)]
        + [fmt(d[s]["n_per_jet"], 3) for s in ("rsd",) + MODELS]
        + [fmt(floor, 3)]
        + [(fmt(d[s]["rel_dev_r"], 3) if ok else "n/s") for s in MODELS]
    )
RATE_TABLE = md_table(
    ["observable", "truth", "rsd", "map", "mbr", "post", "noise floor",
     "map/rsd", "mbr/rsd", "post/rsd"], rate_rows)

print("HEADLINE  -- geometric-mean improvement ratio over SCOREABLE rows only, and\n"
      "            how many of them were beaten (ratio < 1 = better than plain RSD)\n")
print(HEADLINE)
for m, mlab in METRICS:
    n_ns = sum(1 for _, d in ROWS if not d["scoreable"][m])
    print(f"\n\n{mlab}  (distance to truth; lower is better; "
          f"{n_ns}/{len(ROWS)} rows [n/s] = plain RSD is inside the noise floor)\n")
    print(TABLES[m])
print("\n\nRATE  (emissions per jet above a kt cut; the ratio columns compare the\n"
      "       RELATIVE deviation from truth, which the three shape distances above\n"
      "       cannot see because they normalise it away)\n")
print(RATE_TABLE)

In [ ]:
if WRITE_ARTIFACTS:
    out_dir = CKPT.resolve().parent
    metrics = {
        "model": info["model_name"],
        "encoder": str(cfg.encoder.name),
        "checkpoint": str(CKPT),
        "aux_features": list(AUX),
        "continuous_coords": bool(CONT),
        "selection": {"require_truth_splitting": bool(REQUIRE_TRUTH_SPLITTING),
                      "population": ("len(x)>0 and len(y)>0 (v1, truth-selected)"
                                     if REQUIRE_TRUTH_SPLITTING
                                     else "len(x)>0 (deployable)"),
                      "n_in": int(_n_in), "n_kept": int(len(jets)),
                      "p_truth_empty": float(np.mean(NSPL["truth"] == 0)),
                      "map_allow_empty": bool(MAP_ALLOW_EMPTY)},
        "empty_tree": EMPTY_TREE,          # section 5a; computed there, kept here
        "data": {
            "source": SOURCE, "path": str(ROOT_PATH),
            "generator": str(jets[0].get("generator", "n/a")),
            "z_cut": Z_CUT, "beta": BETA,
            "kt_floor": float(jets[0].get("kt_floor", float("nan"))),
            # The OFF-SPINE floor. Two files agreeing on kt_floor can still be
            # different aux samples (docs/PLAN_prod_test_v0.md), so recording only
            # kt_floor makes an asymmetric-file run indistinguishable from a symmetric
            # one in this artifact.
            "kt_floor_sec": float(jets[0].get("kt_floor_sec", float("nan"))),
            "pt_var": PT_VAR, "pt_min": PT_MIN, "pt_max": PT_MAX,
            "n_eval_jets": int(N), "sum_w": float(w_jet.sum()),
            "eff_n": float(w_jet.sum() ** 2 / (w_jet ** 2).sum()),
        },
        "decode": {**DECODE, "point_estimator": "map+mbr",
                   "mbr_backend": MBR_BACKEND,
                   "mbr_n_candidates": MBR_N_CANDIDATES,
                   "length_floor_quantile": LENGTH_FLOOR_QUANTILE},
        "n_draws": int(K_DRAWS),
        "fiducial_window": {"ln_invdelta": list(geom.ln_invdelta_range),
                            "ln_kt": list(geom.ln_kt_range),
                            "ln_z_floor": LNZ_FLOOR},
        "support": SUPPORT,
        "mean_multiplicity": {s: float(NSPL[s].mean()) for s in SERIES},
        "mbr_risk_mean": float(np.nanmean(mbr_risk)) if np.isfinite(mbr_risk).any() else None,
        "scoreability": {"n_boot": int(N_BOOT), "floor_percentile": int(FLOOR_PCT),
                         "n_rows": len(ROWS),
                         "n_scoreable": {m: int(sum(1 for _, d in ROWS
                                                    if d["scoreable"][m]))
                                         for m, _ in METRICS}},
        "headline": {
            s: {m: {"gmean_ratio": gmean(scored(s, m)),
                    "n_better": int(sum(1 for x in scored(s, m)
                                        if np.isfinite(x) and x < 1.0)),
                    "n_scored": int(sum(1 for x in scored(s, m) if np.isfinite(x)))}
                for m, _ in METRICS}
            for s in MODELS
        },
        "observables": {name: d for name, d in ROWS},
        "rates": {name: {"truth_per_jet": ref, "noise_floor": floor,
                         "scoreable": ok, **d}
                  for name, ref, floor, ok, d in RATE_ROWS},
    }
    p = save_metrics(metrics, out_dir / "dist_closure_metrics.json")
    body = ["# Lund distribution closure", "",
            f"`{CKPT}`  --  {SOURCE} `{ROOT_PATH}`, {N} jets, K={K_DRAWS} draws", "",
            "## Headline (geometric-mean improvement ratio vs plain RSD)", "", HEADLINE]

    # Section 5a. It goes directly under the headline because it is the observable v2
    # exists to expose, and the one no distance row below can see.
    et = ["| series | P(n=0) | vs truth | P(n_hat=0) on truth-empty jets |",
          "|--------|--------|----------|--------------------------------|",
          f"| truth | {100 * EMPTY_TREE['p_truth']:.2f}% | -- | -- |"]

    def et_row(label, p_, rec):
        ratio = f"{p_ / EMPTY_TREE['p_truth']:.2f}x" if EMPTY_TREE["p_truth"] > 0 else "--"
        cond = "--" if rec is None else format(100 * rec, ".1f") + "%"
        et.append(f"| {label} | {100 * p_:.2f}% | {ratio} | {cond} |")

    for _s in SERIES:
        if _s != "truth":
            et_row(_s, EMPTY_TREE["p_pred"][_s], EMPTY_TREE["recall"][_s])
    if EMPTY_TREE["map_no_floor"]:
        et_row("map (no floor)", EMPTY_TREE["map_no_floor"]["p_pred"],
               EMPTY_TREE["map_no_floor"]["recall"])
    if EMPTY_TREE["gate"]:
        et_row(f"gate tau={EMPTY_TREE['gate']['tau']:.3f}",
               EMPTY_TREE["gate"]["p_pred"], EMPTY_TREE["gate"]["recall"])
    body += ["", "## Empty parton tree", "", "\n".join(et), "",
             f"MBR backend `{MBR_BACKEND}`: the MBR row moves with it (the imbalance "
             "penalty makes an empty cloud near-maximal risk), the others do not."]
    _g = EMPTY_TREE["gate"]
    if _g and _g["precision"] is not None and _g["recall"] is not None:
        body += ["", f"Gate `decode.empty_threshold = {_g['tau']:.4f}`"
                 + (", RATE-MATCHED on this sample -- fit it on held-out jets before "
                    "quoting it" if _g["rate_matched_on_this_sample"] else "")
                 + f": precision {_g['precision']:.3f}, recall {_g['recall']:.3f}."]
    if EMPTY_TREE["q0"]["auc"] is not None:
        body += ["", f"`q(0|x)` AUC {EMPTY_TREE['q0']['auc']:.3f}, mean "
                 f"{EMPTY_TREE['q0']['mean']:.4f} against a true rate of "
                 f"{EMPTY_TREE['p_truth']:.4f} -- the belief itself, before any decode."]
    for m, mlab in METRICS:
        body += ["", f"## {mlab}", "", TABLES[m]]
    body += ["", "## Rate (emissions per jet above a kt cut)", "", RATE_TABLE]
    q = out_dir / "dist_closure_table.md"
    q.write_text("\n".join(body) + "\n")
    print(f"wrote {p}")
    print(f"wrote {q}")
else:
    print("WRITE_ARTIFACTS is off -- nothing written")

## Reading the results

**Do not compare these numbers to v1's.** They are computed on a different population —
17% more jets, all of them ones whose answer is the empty tree — so every distance, ratio
and rate moves. v1's are not superseded so much as *scoped*: they describe jets that have
parton-level substructure, which is a legitimate question, just not the one an analysis
gets to ask. Where the two disagree, v2 is the one that describes what happens on data.
Run v2 with `REQUIRE_TRUTH_SPLITTING = True` if you want them on the same footing.

**Neither point estimator can win §5a, for two unrelated reasons.** The MAP's
$P(\hat n = 0)$ is zero because `decode.min_emissions = 1` forbids it — but lifting the
floor changes nothing, because with a multiplicity head the MAP is
$\arg\max_n q(n \mid x)$ and the peak lands at $n{=}0$ essentially never; a model can hold
16% of its length mass at zero and still never put its mode there. MBR is mode-free and
floor-free, yet under the default `pot` metric it also returns ~0%, because the
perturbative-Lund EMD's imbalance penalty makes an empty cloud close to the worst possible
answer. Both are ceilings of the *decode*, not errors of the model, which carries real
information here (AUC 0.77). Getting these jets right needs a decision rule that can
express "nothing" — thresholding $q(0 \mid x)$, say — rather than a different floor or a
different beam width. Note also that this one column swings with `MBR_BACKEND`
(`surrogate` reads ~57% where `pot` reads ~0.2%); the shape panels do not.

**The ratio is the verdict, the panels are the diagnosis.** A geometric-mean ratio below 1
with most observables beaten means the model genuinely moves the hadron-level distribution
toward parton level. Above 1 means plain RSD was already better and the model is adding
noise.

**Six things that are not model failures.**

0. **An `[n/s]` row.** Plain RSD is already within truth-vs-truth noise there, so
   hadronisation left no shape difference to recover. Expect $\ln z$ and $\psi$ rows to fall
   out this way — the first is squeezed between two hard walls, the second is uniform by
   symmetry at both levels. Raising `N_JETS` lowers the floors and brings marginal rows back
   into scope; it will never rescue a row where the two levels genuinely agree.

1. **Mode shrinkage.** MAP and MBR are per-jet argmaxes; their populations are narrower
   than truth by construction. The `post` series is the calibrated comparator — judge the
   model by it and the *estimators* by their gap to it.
2. **The $\psi$ panel for MAP/MBR** is the von Mises mode, which is near-arbitrary at small
   $\kappa$. Read it only in light of the $\kappa$ histogram in §7.
3. **Out-of-window truth.** The model cannot emit outside the geometry's $(0,6)^2$; the §5
   fraction is an irreducible floor. That is why the distances use the fiducial window.
4. **Plain RSD is a strong baseline** for hard emissions — hadron level already tracks
   parton level there. Expect ratios near 1 in the `*_hard` quadrant rows and below 1 in
   `*_soft`. The pooled row hides this; the quadrant rows are where the model earns its keep.
5. **A family with no coordinate density** (`ar_junipr_v1` only — every other family
   draws $\ln z$ and $\psi$ through `sample_coordinates`) leaves those entries as
   placeholders, so those panels drop the model series rather than plotting a filler
   constant as a prediction.

**One thing that *is* a model failure:** a non-zero soft-drop violation fraction in §5. The
$\ln z$ head is an unbounded normal and knows nothing about the grooming boundary, so any
mass below it is unphysical by definition.

**Re-running.** Point `CKPT_PATH` at another checkpoint, or set `PT_MIN`/`PT_MAX` to sweep
the `docs/PLAN_ProductionAssessment.md` §7 windows ($[100,150)$, $[150,250)$,
$[250,\infty)$ GeV) — the artifacts land beside each checkpoint, so the JSONs from several
runs can be collated into that plan's `assessment_table.md`.

**If this proves useful**, two things want promoting out of the notebook: a
`sample_coordinates(xf, nx, cells)` method on the `PosteriorModel` contract (removing the
only private-API reach here and making the posterior series family-agnostic), and an
`eval/distribution.py` holding `w1` / `ks_w` / `chi2_ndf`, which would let `cmd_eval` emit
population metrics into `eval_metrics.json` directly.